## 0. Preparación

In [ ]:
import json
import random
import re
import time
from datetime import date, datetime, timezone
from enum import Enum
from pathlib import Path
from typing import Annotated, Literal, TypeAlias

import pandas as pd
from pydantic import (
    BaseModel,
    ConfigDict,
    Field,
    StringConstraints,
    field_validator,
    model_validator,
)
from pydantic_ai import Agent

from collections import Counter
from typing_extensions import Self

from renewables_permitting.utils import (
    enum_value,
    normalize_text_or_none,
    safe_str,
    save_parquet,
    to_date_or_none,
    validate_required_columns,
)

BASE_URL = "https://www.boe.es/datosabiertos/api/boe/sumario"

# PROJECT_ROOT = Path(__file__).resolve().parents[2]  # fuera del notebook
PROJECT_ROOT = Path.cwd().parent  # dentro del notebook

DATA_DIR = PROJECT_ROOT / "data"

BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

# BRONZE
BOE_DOCS_XML_DIR = BRONZE_DIR / "boe_docs_xml"


# SILVER: candidatos BOE
BOE_CANDIDATES_PATH = SILVER_DIR / "boe_candidates" / "boe_candidates_normalized.parquet"
BOE_CANDIDATES_DOCS_TEXT_PATH = SILVER_DIR / "boe_candidates_docs_text" / "boe_candidates_docs_text.parquet"

# SILVER: dimensiones
DIM_MUNICIPALITIES_PATH = SILVER_DIR / "dimensions" / "dim_municipalities.parquet"

# SILVER: extracción IA
SILVER_BOE_AI_DIR = SILVER_DIR / "boe_ai"

BOE_AI_EXTRACTIONS_PATH = SILVER_BOE_AI_DIR / "boe_ai_extractions.parquet"

PUBLICATION_EVENTS_PATH = SILVER_BOE_AI_DIR / "publication_events.parquet"
ADMINISTRATIVE_ACTIONS_PATH = SILVER_BOE_AI_DIR / "administrative_actions.parquet"
PROJECT_MENTIONS_PATH = SILVER_BOE_AI_DIR / "project_mentions.parquet"
PROJECT_TECHNICAL_ATTRIBUTES_PATH = SILVER_BOE_AI_DIR / "project_technical_attributes.parquet"
PROJECT_PARTICIPANTS_PATH = SILVER_BOE_AI_DIR / "project_participants.parquet"
PROJECT_LOCATIONS_PATH = SILVER_BOE_AI_DIR / "project_locations.parquet"
PROJECT_ALIASES_PATH = SILVER_BOE_AI_DIR / "project_aliases.parquet"
ASSOCIATED_INFRASTRUCTURE_PATH = SILVER_BOE_AI_DIR / "associated_infrastructure.parquet"

- La **pregunta central** del TFM es: ¿En qué estado administrativo se encuentra cada proyecto energético según las publicaciones del BOE?

- La extracción debería centrarse en tres problemas:
    1. Identificar el proyecto mencionado.
    2. Determinar qué acto administrativo publica el BOE.
    3. Integrar cronológicamente ese acto con publicaciones anteriores del mismo proyecto. (En siguiente paso fuera de la extracción con IA)


- Enum solo define valores permitidos.

- El idioma. Texto fuente del BOE → español. Categoría normalizada del sistema → inglés.
    - Python API names are in English.
    - Spanish BOE legal taxonomy values are stored as Spanish normalized slugs. El valor almacenado será jurídicamente próximo al BOE: "declaracion_impacto_ambiental"

### Tipos reutilizables y modelo base

In [ ]:
NonEmptyText: TypeAlias = Annotated[
    str,
    StringConstraints(
        strip_whitespace=True,
        min_length=1,
    ),
]


AssetRef: TypeAlias = Annotated[
    str,
    StringConstraints(
        strip_whitespace=True,
        pattern=r"^asset_[1-9][0-9]*$",
    ),
]


BOEId: TypeAlias = Annotated[
    str,
    StringConstraints(
        strip_whitespace=True,
        pattern=r"^BOE-[A-Z]-[0-9]{4}-[0-9]+$",
    ),
]


class ContractModel(BaseModel):
    """
    Modelo base de los contratos de extracción.

    extra="forbid" impide que el modelo añada campos no definidos.
    str_strip_whitespace elimina espacios exteriores de los textos.
    """

    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
        validate_default=True,
    )


def _deduplicate_strings(values: list[str]) -> list[str]:
    """
    Elimina duplicados preservando:

    - el orden original;
    - la grafía de la primera aparición.

    La comparación no distingue entre mayúsculas y minúsculas.
    """

    seen: set[str] = set()
    result: list[str] = []

    for value in values:
        key = value.casefold()

        if key not in seen:
            seen.add(key)
            result.append(value)

    return result

### Clasificación documental

In [ ]:
class ClassificationStatus(str, Enum):
    """
    Estado interno de la clasificación realizada por la IA.

    Sus valores pertenecen al control del pipeline, no a la taxonomía jurídica
    del BOE, por lo que se mantienen en inglés.
    """

    CLASSIFIED = "classified"
    UNCERTAIN = "uncertain"


class DocumentScope(str, Enum):
    """Alcance energético de la publicación."""

    NOT_ENERGY_RELEVANT = "not_energy_relevant"
    ENERGY_GENERAL = "energy_general"
    ENERGY_PROJECT_SPECIFIC = "energy_project_specific"

### Activos energéticos

In [ ]:
class EnergyInstallationType(str, Enum):
    """
    Taxonomía controlada del tipo principal de instalación.

    Las infraestructuras eléctricas solo se clasifican como activos cuando
    constituyen el objeto principal autónomo de la publicación.
    """

    PHOTOVOLTAIC = "fotovoltaica"
    WIND = "eolica"
    CONCENTRATED_SOLAR_POWER = "termosolar"
    HYDROPOWER = "hidroelectrica"
    GEOTHERMAL = "geotermica"
    BIOMASS = "biomasa"
    BIOGAS = "biogas"
    GREEN_HYDROGEN = "hidrogeno_verde"

    ENERGY_STORAGE = "almacenamiento"

    EVACUATION_INFRASTRUCTURE = "infraestructura_evacuacion"
    ELECTRICAL_SUBSTATION = "subestacion_electrica"
    POWER_LINE = "linea_electrica"

    OTHER = "otra"
    UNKNOWN = "desconocido"

class ProjectRoleInEvent(str, Enum):
    """
    Papel que desempeña el activo en esta publicación concreta.

    No representa el estado consolidado del proyecto.
    """

    PRIMARY_SUBJECT = "objeto_principal"
    EXISTING_REFERENCE = "referencia_existente"
    ASSOCIATED_REFERENCE = "referencia_asociada"
    UNKNOWN = "desconocido"

### Menciones técnicas literales

In [ ]:
class TechnicalAttributeType(str, Enum):
    """Tipo semántico de una magnitud o característica técnica."""

    INSTALLED_POWER = "potencia_instalada"
    PEAK_POWER = "potencia_pico"
    STORAGE_POWER = "potencia_almacenamiento"
    STORAGE_CAPACITY = "capacidad_almacenamiento"

    UNIT_COUNT = "numero_unidades"
    UNIT_POWER = "potencia_unitaria"

    VOLTAGE = "tension"

    OTHER = "otra"
    UNKNOWN = "desconocido"


class TechnicalMention(ContractModel):
    """
    Expresión técnica literal atribuida a un activo.

    La IA identifica el tipo de magnitud y conserva su expresión original.
    La conversión de unidades, los cálculos y las comprobaciones de coherencia
    se realizan posteriormente mediante código determinista.
    """

    attribute_type: TechnicalAttributeType

    value_raw: NonEmptyText = Field(
        description=(
            "Valor o expresión técnica tal como aparece en el BOE. "
            "Ejemplos: '31,172 MW', '14 aerogeneradores de 2.000 kW' "
            "o '26,36 MW / 52,72 MWh'."
        )
    )

    evidence: NonEmptyText = Field(
        description=(
            "Fragmento suficiente del BOE que permite comprobar la mención."
        )
    )


class EnergyAssetMention(ContractModel):
    """
    Instalación energética identificable mencionada en el evento.

    Se extraen únicamente:

    - activos objeto principal del procedimiento;
    - activos existentes necesarios para interpretar una hibridación,
      modificación, repotenciación, ampliación o sustitución;
    - infraestructura eléctrica cuando sea el objeto principal autónomo.

    No deben extraerse como activos independientes componentes internos como
    aerogeneradores, módulos, posiciones, centros de transformación o tramos
    auxiliares.
    """

    local_asset_ref: AssetRef = Field(
        description=(
            "Referencia local dentro del PublicationEvent: asset_1, asset_2, "
            "etc. No constituye la identidad persistente o consolidada."
        )
    )

    name_raw: NonEmptyText | None = Field(
        default=None,
        description=(
            "Nombre literal del activo. Debe ser None cuando el BOE no "
            "proporcione una denominación identificable."
        ),
    )

    aliases_raw: list[NonEmptyText] = Field(
        default_factory=list,
        description=(
            "Denominaciones alternativas expresamente presentes en el BOE. "
            "No deben generarse variantes normalizadas o inferidas."
        ),
    )

    installation_type: EnergyInstallationType
    role_in_event: ProjectRoleInEvent

    technical_mentions: list[TechnicalMention] = Field(
        default_factory=list,
        description=(
            "Características técnicas literales atribuibles al activo."
        ),
    )

    technical_summary: NonEmptyText | None = Field(
        default=None,
        description=(
            "Síntesis técnica breve y prudente. No sustituye a "
            "technical_mentions ni debe introducir cifras no documentadas."
        ),
    )

    evidence: NonEmptyText = Field(
        description=(
            "Fragmento que identifica el activo y permite verificar su papel."
        )
    )

    @field_validator("aliases_raw")
    @classmethod
    def deduplicate_aliases(
        cls,
        values: list[str],
    ) -> list[str]:
        """Elimina aliases repetidos sin alterar su primera grafía."""

        return _deduplicate_strings(values)

    @model_validator(mode="after")
    def remove_name_from_aliases(self) -> Self:
        """
        Evita que el nombre principal vuelva a aparecer dentro de aliases_raw.

        Se trata de una limpieza determinista y no provoca un nuevo intento de
        extracción por una duplicación menor.
        """

        if self.name_raw is None:
            return self

        name_key = self.name_raw.casefold()

        self.aliases_raw = [
            alias
            for alias in self.aliases_raw
            if alias.casefold() != name_key
        ]

        return self

### Actuaciones administrativas

In [ ]:
class AdministrativeActionType(str, Enum):
    """
    Acto, trámite o producto administrativo publicado.

    No representa por sí mismo la decisión o resultado del acto.
    """

    # Inicio y tramitación
    APPLICATION_SUBMISSION = "solicitud_tramitacion"
    ENVIRONMENTAL_APPLICATION_SUBMISSION = (
        "solicitud_tramitacion_ambiental"
    )
    DOCUMENTATION_CORRECTION = "subsanacion_documentacion"
    REQUIREMENTS_VERIFICATION = "verificacion_requisitos_tramitacion"
    ERROR_CORRECTION = "correccion_errores"

    # Información pública
    PUBLIC_INFORMATION = "informacion_publica"

    # Evaluación ambiental
    ENVIRONMENTAL_IMPACT_ASSESSMENT = (
        "evaluacion_impacto_ambiental"
    )
    ENVIRONMENTAL_IMPACT_STATEMENT = (
        "declaracion_impacto_ambiental"
    )
    ENVIRONMENTAL_IMPACT_REPORT = (
        "informe_impacto_ambiental"
    )
    ENVIRONMENTAL_AFFECTATION_DETERMINATION_REPORT = (
        "informe_determinacion_afeccion_ambiental"
    )

    # Autorizaciones energéticas
    PRIOR_ADMINISTRATIVE_AUTHORIZATION = (
        "autorizacion_administrativa_previa"
    )
    CONSTRUCTION_ADMINISTRATIVE_AUTHORIZATION = (
        "autorizacion_administrativa_construccion"
    )
    OPERATING_AUTHORIZATION = "autorizacion_explotacion"

    # Utilidad pública y expropiación
    PUBLIC_UTILITY_DECLARATION = "declaracion_utilidad_publica"
    FORCED_EXPROPRIATION = "expropiacion_forzosa"
    AFFECTED_ASSETS_AND_RIGHTS_LIST = (
        "relacion_bienes_derechos_afectados"
    )
    PRIOR_OCCUPATION_RECORDS = (
        "levantamiento_actas_previas_ocupacion"
    )
    OCCUPATION_RECORDS = "actas_ocupacion"

    # Modificaciones y terminación
    AUTHORIZATION_MODIFICATION = "modificacion_autorizacion"
    DEADLINE_EXTENSION = "prorroga"
    OWNERSHIP_CHANGE = "cambio_titularidad"
    PROCEDURE_TERMINATION = "terminacion_procedimiento"

    OTHER = "otro"
    UNKNOWN = "desconocido"


class AdministrativeDecision(str, Enum):
    """Resultado producido por una actuación administrativa."""

    # Inicio y tramitación
    REQUESTED = "solicitado"
    CORRECTED = "subsanado"
    REQUIREMENTS_VERIFIED = "requisitos_verificados"
    RECTIFIED = "rectificado"

    # Información pública
    SUBMITTED_TO_PUBLIC_INFORMATION = (
        "sometido_informacion_publica"
    )
    ANNOUNCED = "convocado"

    # Evaluación ambiental
    FORMULATED = "formulado"
    FAVORABLE = "favorable"
    UNFAVORABLE = "desfavorable"

    NO_SIGNIFICANT_ADVERSE_ENVIRONMENTAL_EFFECTS = (
        "sin_efectos_adversos_significativos"
    )
    ORDINARY_ENVIRONMENTAL_ASSESSMENT_REQUIRED = (
        "requiere_evaluacion_ambiental_ordinaria"
    )
    FURTHER_ENVIRONMENTAL_ASSESSMENT_REQUIRED = (
        "requiere_evaluacion_ambiental_adicional"
    )
    FURTHER_ENVIRONMENTAL_ASSESSMENT_NOT_REQUIRED = (
        "no_requiere_evaluacion_ambiental_adicional"
    )

    # Resoluciones
    AUTHORIZED = "autorizado"
    DECLARED = "declarado"
    MODIFIED = "modificado"
    EXTENDED = "prorrogado"

    DENIED = "denegado"
    CLOSED = "archivado"
    WITHDRAWN = "desistido"
    INADMISSIBLE = "inadmitido"

    OTHER = "otro"
    UNKNOWN = "desconocido"


class AdministrativeActionScope(str, Enum):
    """
    Alcance documental de una actuación.

    SPECIFIC_ASSETS:
        el BOE permite identificar inequívocamente los activos afectados.

    EVENT_LEVEL:
        la actuación afecta al expediente o evento completo y no debe
        descomponerse artificialmente.

    UNKNOWN:
        el alcance no puede establecerse con suficiente seguridad.
    """

    SPECIFIC_ASSETS = "activos_especificos"
    EVENT_LEVEL = "evento_completo"
    UNKNOWN = "desconocido"


# Combinaciones conocidas y semánticamente compatibles.
#
# OTHER y UNKNOWN se admiten siempre como mecanismos de escape controlado.
# La tabla puede ampliarse tras auditar nuevos tipos documentales.
_ALLOWED_DECISIONS_BY_ACTION_TYPE: dict[
    AdministrativeActionType,
    set[AdministrativeDecision],
] = {
    AdministrativeActionType.APPLICATION_SUBMISSION: {
        AdministrativeDecision.REQUESTED,
    },
    AdministrativeActionType.ENVIRONMENTAL_APPLICATION_SUBMISSION: {
        AdministrativeDecision.REQUESTED,
    },
    AdministrativeActionType.DOCUMENTATION_CORRECTION: {
        AdministrativeDecision.CORRECTED,
    },
    AdministrativeActionType.REQUIREMENTS_VERIFICATION: {
        AdministrativeDecision.REQUIREMENTS_VERIFIED,
    },
    AdministrativeActionType.ERROR_CORRECTION: {
        AdministrativeDecision.RECTIFIED,
    },
    AdministrativeActionType.PUBLIC_INFORMATION: {
        AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION,
        AdministrativeDecision.ANNOUNCED,
    },
    AdministrativeActionType.ENVIRONMENTAL_IMPACT_ASSESSMENT: {
        AdministrativeDecision.REQUESTED,
        AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION,
    },
    AdministrativeActionType.ENVIRONMENTAL_IMPACT_STATEMENT: {
        AdministrativeDecision.FORMULATED,
        AdministrativeDecision.FAVORABLE,
        AdministrativeDecision.UNFAVORABLE,
    },
    AdministrativeActionType.ENVIRONMENTAL_IMPACT_REPORT: {
        AdministrativeDecision.FORMULATED,
        AdministrativeDecision.NO_SIGNIFICANT_ADVERSE_ENVIRONMENTAL_EFFECTS,
        AdministrativeDecision.ORDINARY_ENVIRONMENTAL_ASSESSMENT_REQUIRED,
    },
    AdministrativeActionType.ENVIRONMENTAL_AFFECTATION_DETERMINATION_REPORT: {
        AdministrativeDecision.FORMULATED,
        AdministrativeDecision.FAVORABLE,
        AdministrativeDecision.UNFAVORABLE,
        AdministrativeDecision.FURTHER_ENVIRONMENTAL_ASSESSMENT_REQUIRED,
        AdministrativeDecision.FURTHER_ENVIRONMENTAL_ASSESSMENT_NOT_REQUIRED,
    },
    AdministrativeActionType.PRIOR_ADMINISTRATIVE_AUTHORIZATION: {
        AdministrativeDecision.REQUESTED,
        AdministrativeDecision.AUTHORIZED,
        AdministrativeDecision.DENIED,
    },
    AdministrativeActionType.CONSTRUCTION_ADMINISTRATIVE_AUTHORIZATION: {
        AdministrativeDecision.REQUESTED,
        AdministrativeDecision.AUTHORIZED,
        AdministrativeDecision.DENIED,
    },
    AdministrativeActionType.OPERATING_AUTHORIZATION: {
        AdministrativeDecision.REQUESTED,
        AdministrativeDecision.AUTHORIZED,
        AdministrativeDecision.DENIED,
    },
    AdministrativeActionType.PUBLIC_UTILITY_DECLARATION: {
        AdministrativeDecision.REQUESTED,
        AdministrativeDecision.DECLARED,
        AdministrativeDecision.DENIED,
    },
    AdministrativeActionType.FORCED_EXPROPRIATION: {
        AdministrativeDecision.REQUESTED,
        AdministrativeDecision.AUTHORIZED,
        AdministrativeDecision.DECLARED,
        AdministrativeDecision.ANNOUNCED,
    },
    AdministrativeActionType.AFFECTED_ASSETS_AND_RIGHTS_LIST: {
        AdministrativeDecision.SUBMITTED_TO_PUBLIC_INFORMATION,
        AdministrativeDecision.ANNOUNCED,
    },
    AdministrativeActionType.PRIOR_OCCUPATION_RECORDS: {
        AdministrativeDecision.ANNOUNCED,
        AdministrativeDecision.FORMULATED,
    },
    AdministrativeActionType.OCCUPATION_RECORDS: {
        AdministrativeDecision.ANNOUNCED,
        AdministrativeDecision.FORMULATED,
    },
    AdministrativeActionType.AUTHORIZATION_MODIFICATION: {
        AdministrativeDecision.REQUESTED,
        AdministrativeDecision.AUTHORIZED,
        AdministrativeDecision.MODIFIED,
        AdministrativeDecision.DENIED,
    },
    AdministrativeActionType.DEADLINE_EXTENSION: {
        AdministrativeDecision.REQUESTED,
        AdministrativeDecision.EXTENDED,
        AdministrativeDecision.DENIED,
    },
    AdministrativeActionType.OWNERSHIP_CHANGE: {
        AdministrativeDecision.REQUESTED,
        AdministrativeDecision.AUTHORIZED,
        AdministrativeDecision.DENIED,
    },
    AdministrativeActionType.PROCEDURE_TERMINATION: {
        AdministrativeDecision.CLOSED,
        AdministrativeDecision.WITHDRAWN,
        AdministrativeDecision.INADMISSIBLE,
    },
}


class AdministrativeAction(ContractModel):
    """
    Acto administrativo publicado y activos sobre los que recae.

    Las instalaciones mencionadas únicamente como contexto no deben incluirse
    en affected_asset_refs.
    """

    action_type: AdministrativeActionType
    decision: AdministrativeDecision

    scope: AdministrativeActionScope

    affected_asset_refs: list[AssetRef] = Field(
        default_factory=list,
        description=(
            "Activos afectados directamente. Solo se rellena cuando "
            "scope='activos_especificos'."
        ),
    )

    evidence: NonEmptyText

    @field_validator("affected_asset_refs")
    @classmethod
    def deduplicate_asset_refs(
        cls,
        values: list[str],
    ) -> list[str]:
        return _deduplicate_strings(values)

    @model_validator(mode="after")
    def validate_scope(self) -> Self:
        """Impide combinaciones contradictorias entre alcance y referencias."""

        if (
            self.scope
            == AdministrativeActionScope.SPECIFIC_ASSETS
            and not self.affected_asset_refs
        ):
            raise ValueError(
                "Una actuación con scope='activos_especificos' debe "
                "referenciar al menos un activo."
            )

        if (
            self.scope
            != AdministrativeActionScope.SPECIFIC_ASSETS
            and self.affected_asset_refs
        ):
            raise ValueError(
                "Solo las actuaciones con scope='activos_especificos' "
                "pueden contener affected_asset_refs."
            )

        return self

    @model_validator(mode="after")
    def validate_action_decision_combination(self) -> Self:
        """
        Rechaza combinaciones conocidas como incoherentes.

        Los valores OTHER y UNKNOWN permanecen disponibles cuando el documento
        no encaje con seguridad en la taxonomía controlada.
        """

        if self.action_type in {
            AdministrativeActionType.OTHER,
            AdministrativeActionType.UNKNOWN,
        }:
            return self

        if self.decision in {
            AdministrativeDecision.OTHER,
            AdministrativeDecision.UNKNOWN,
        }:
            return self

        allowed_decisions = _ALLOWED_DECISIONS_BY_ACTION_TYPE.get(
            self.action_type
        )

        if (
            allowed_decisions is not None
            and self.decision not in allowed_decisions
        ):
            allowed_values = sorted(
                decision.value
                for decision in allowed_decisions
            )

            raise ValueError(
                "Combinación administrativa incoherente: "
                f"{self.action_type.value!r} + {self.decision.value!r}. "
                f"Decisiones permitidas: {allowed_values}."
            )

        return self

### Afirmaciones documentales

In [ ]:
class DocumentaryClaimType(str, Enum):
    """
    Tipos de afirmaciones documentales complementarias.

    Los datos técnicos no se incluyen aquí porque permanecen asociados
    directamente al activo mediante TechnicalMention.
    """

    PARTICIPANT = "participante"
    ADMINISTRATIVE_LOCATION = "localizacion_administrativa"
    ASSOCIATED_INFRASTRUCTURE = "infraestructura_asociada"
    ASSET_RELATION = "relacion_entre_activos"
    OTHER = "otro"


class DocumentaryClaimScope(str, Enum):
    """Alcance de una afirmación documental."""

    SPECIFIC_ASSETS = "activos_especificos"
    EVENT_LEVEL = "evento_completo"
    UNKNOWN = "desconocido"


class ParticipantRole(str, Enum):
    """Rol de una entidad respecto de los activos o del expediente."""

    PROMOTER = "promotor"
    CO_PROMOTER = "copromotor"
    HOLDER = "titular"
    OPERATOR = "operador"
    APPLICANT = "solicitante"

    TRANSFEROR = "cedente"
    TRANSFEREE = "cesionario"

    OTHER = "otro"
    UNKNOWN = "desconocido"


class AdministrativeLocationLevel(str, Enum):
    """Nivel administrativo del nombre territorial extraído."""

    MUNICIPALITY = "municipio"
    PROVINCE = "provincia"
    AUTONOMOUS_COMMUNITY = "comunidad_autonoma"


class AssociatedInfrastructureFeature(str, Enum):
    """
    Rasgos detectados en la infraestructura auxiliar.

    La lista puede combinar, por ejemplo, infraestructura de evacuación,
    subestación e infraestructura compartida.
    """

    EVACUATION_INFRASTRUCTURE = "infraestructura_evacuacion"
    ELECTRICAL_SUBSTATION = "subestacion_electrica"
    POWER_LINE = "linea_electrica"
    GRID_CONNECTION = "conexion_red"
    SHARED_INFRASTRUCTURE = "infraestructura_compartida"
    OTHER = "otra"


class EnergyAssetRelationType(str, Enum):
    """
    Relación dirigida entre dos activos.

    El activo de subject_asset_refs es el origen de la relación y
    related_asset_ref es el destino o activo de referencia.
    """

    HYBRIDIZES_WITH = "hibrida_con"
    ADDS_STORAGE_TO = "incorpora_almacenamiento_a"
    MODIFIES = "modifica"
    REPOWERS = "repotencia"
    REPLACES = "sustituye_a"
    SHARES_EVACUATION_WITH = "comparte_evacuacion_con"

    OTHER = "otra"
    UNKNOWN = "desconocido"


class DocumentaryClaimBase(ContractModel):
    """
    Campos comunes de toda afirmación documental.

    value_raw conserva la denominación o expresión próxima al texto fuente.
    evidence contiene el fragmento que permite comprobarla.
    """

    claim_type: DocumentaryClaimType
    scope: DocumentaryClaimScope

    subject_asset_refs: list[AssetRef] = Field(
        default_factory=list,
        description=(
            "Activos a los que se atribuye la afirmación. Solo se rellena "
            "cuando scope='activos_especificos'."
        ),
    )

    value_raw: NonEmptyText
    evidence: NonEmptyText

    @field_validator("subject_asset_refs")
    @classmethod
    def deduplicate_subject_refs(
        cls,
        values: list[str],
    ) -> list[str]:
        return _deduplicate_strings(values)

    @model_validator(mode="after")
    def validate_scope(self) -> Self:
        """Comprueba la coherencia entre alcance y referencias."""

        if (
            self.scope == DocumentaryClaimScope.SPECIFIC_ASSETS
            and not self.subject_asset_refs
        ):
            raise ValueError(
                "Una afirmación con scope='activos_especificos' debe "
                "referenciar al menos un activo."
            )

        if (
            self.scope != DocumentaryClaimScope.SPECIFIC_ASSETS
            and self.subject_asset_refs
        ):
            raise ValueError(
                "Solo las afirmaciones con scope='activos_especificos' "
                "pueden contener subject_asset_refs."
            )

        return self


class ParticipantClaim(DocumentaryClaimBase):
    """
    Participante identificado en el BOE.

    value_raw contiene el nombre literal de la entidad. La normalización y
    consolidación societaria se realizan después.
    """

    claim_type: Literal[
        DocumentaryClaimType.PARTICIPANT
    ] = DocumentaryClaimType.PARTICIPANT

    participant_role: ParticipantRole


class AdministrativeLocationClaim(DocumentaryClaimBase):
    """
    Nombre administrativo todavía no resuelto contra referencias del
    Instituto Nacional de Estadística (INE).

    value_raw contiene el municipio, provincia o comunidad autónoma tal como
    aparece en el BOE.
    """

    claim_type: Literal[
        DocumentaryClaimType.ADMINISTRATIVE_LOCATION
    ] = DocumentaryClaimType.ADMINISTRATIVE_LOCATION

    location_level: AdministrativeLocationLevel

    province_hint_raw: NonEmptyText | None = None
    autonomous_community_hint_raw: NonEmptyText | None = None


class AssociatedInfrastructureClaim(DocumentaryClaimBase):
    """
    Resumen no exhaustivo de infraestructura auxiliar.

    Cuando una línea o subestación sea el objeto principal autónomo del BOE,
    deberá extraerse como EnergyAssetMention y no como esta afirmación.
    """

    claim_type: Literal[
        DocumentaryClaimType.ASSOCIATED_INFRASTRUCTURE
    ] = DocumentaryClaimType.ASSOCIATED_INFRASTRUCTURE

    infrastructure_features: list[
        AssociatedInfrastructureFeature
    ] = Field(
        min_length=1,
        description=(
            "Uno o varios rasgos controlados detectados en la infraestructura."
        ),
    )

    @field_validator("infrastructure_features")
    @classmethod
    def deduplicate_features(
        cls,
        values: list[AssociatedInfrastructureFeature],
    ) -> list[AssociatedInfrastructureFeature]:
        return list(dict.fromkeys(values))


class EnergyAssetRelationClaim(DocumentaryClaimBase):
    """
    Relación dirigida y explícita entre dos activos del mismo evento.

    Ejemplo:

        asset_1 = nuevo BESS
        asset_2 = planta fotovoltaica existente

        asset_1 --hibrida_con--> asset_2
    """

    claim_type: Literal[
        DocumentaryClaimType.ASSET_RELATION
    ] = DocumentaryClaimType.ASSET_RELATION

    related_asset_ref: AssetRef
    relation_type: EnergyAssetRelationType

    @model_validator(mode="after")
    def validate_directed_relation(self) -> Self:
        """Una relación debe tener exactamente un origen y un destino distinto."""

        if self.scope != DocumentaryClaimScope.SPECIFIC_ASSETS:
            raise ValueError(
                "Una relación entre activos debe tener "
                "scope='activos_especificos'."
            )

        if len(self.subject_asset_refs) != 1:
            raise ValueError(
                "Una relación entre activos debe contener exactamente "
                "un activo origen en subject_asset_refs."
            )

        if self.subject_asset_refs[0] == self.related_asset_ref:
            raise ValueError(
                "Una relación entre activos no puede apuntar al mismo activo."
            )

        return self


class OtherDocumentaryClaim(DocumentaryClaimBase):
    """
    Afirmación relevante que no encaja con seguridad en las categorías
    controladas.

    Debe utilizarse de manera excepcional.
    """

    claim_type: Literal[
        DocumentaryClaimType.OTHER
    ] = DocumentaryClaimType.OTHER


# La discriminación por claim_type hace que Pydantic valide directamente contra
# la clase adecuada y evita campos opcionales incompatibles entre sí.
DocumentaryClaim: TypeAlias = Annotated[
    ParticipantClaim
    | AdministrativeLocationClaim
    | AssociatedInfrastructureClaim
    | EnergyAssetRelationClaim
    | OtherDocumentaryClaim,
    Field(discriminator="claim_type"),
]

### Evento publicado

In [ ]:
class PublicationEvent(ContractModel):
    """
    Conjunto coherente de actuaciones administrativas comunicadas por una
    publicación respecto de uno o varios activos energéticos relacionados.

    Una publicación tendrá normalmente un único evento. Solo deben generarse
    varios cuando contenga actuaciones materialmente independientes.
    """

    assets: list[EnergyAssetMention] = Field(
        min_length=1,
        description=(
            "Activos principales y referencias energéticas estrictamente "
            "necesarias para interpretar el evento."
        ),
    )

    administrative_actions: list[AdministrativeAction] = Field(
        min_length=1,
        description="Actos o trámites administrativos publicados.",
    )

    documentary_claims: list[DocumentaryClaim] = Field(
        default_factory=list,
        description=(
            "Participantes, localizaciones, infraestructura auxiliar y "
            "relaciones entre activos."
        ),
    )

    case_file_references: list[NonEmptyText] = Field(
        default_factory=list,
        description=(
            "Referencias literales de expediente presentes en la publicación."
        ),
    )

    event_summary: NonEmptyText = Field(
        description=(
            "Resumen breve del contenido administrativo del evento, en español "
            "y sin inferir el estado consolidado del proyecto."
        )
    )

    @field_validator("case_file_references")
    @classmethod
    def deduplicate_case_file_references(
        cls,
        values: list[str],
    ) -> list[str]:
        return _deduplicate_strings(values)

    @model_validator(mode="after")
    def validate_asset_graph(self) -> Self:
        """
        Comprueba la integridad referencial del pequeño grafo documental.

        Este validador garantiza que las referencias existen, pero no puede
        comprobar por sí solo que la relación semántica extraída sea correcta.
        Esa calidad debe evaluarse mediante ejemplos, auditoría y métricas.
        """

        asset_refs = [
            asset.local_asset_ref
            for asset in self.assets
        ]

        if len(asset_refs) != len(set(asset_refs)):
            raise ValueError(
                "local_asset_ref debe ser único dentro del evento."
            )

        if not any(
            asset.role_in_event == ProjectRoleInEvent.PRIMARY_SUBJECT
            for asset in self.assets
        ):
            raise ValueError(
                "Cada evento debe contener al menos un activo con "
                "role_in_event='objeto_principal'."
            )

        valid_asset_refs = set(asset_refs)

        # Referencias utilizadas por las actuaciones.
        for action in self.administrative_actions:
            missing_refs = (
                set(action.affected_asset_refs)
                - valid_asset_refs
            )

            if missing_refs:
                raise ValueError(
                    "AdministrativeAction contiene referencias a activos "
                    f"inexistentes: {sorted(missing_refs)}."
                )

        # Referencias utilizadas por las afirmaciones documentales.
        relation_keys: set[
            tuple[str, str, EnergyAssetRelationType]
        ] = set()

        for claim in self.documentary_claims:
            missing_refs = (
                set(claim.subject_asset_refs)
                - valid_asset_refs
            )

            if missing_refs:
                raise ValueError(
                    "DocumentaryClaim contiene referencias a activos "
                    f"inexistentes: {sorted(missing_refs)}."
                )

            if isinstance(claim, EnergyAssetRelationClaim):
                if claim.related_asset_ref not in valid_asset_refs:
                    raise ValueError(
                        "EnergyAssetRelationClaim contiene un activo destino "
                        f"inexistente: {claim.related_asset_ref!r}."
                    )

                relation_key = (
                    claim.subject_asset_refs[0],
                    claim.related_asset_ref,
                    claim.relation_type,
                )

                if relation_key in relation_keys:
                    raise ValueError(
                        "Existe una relación entre activos duplicada: "
                        f"{relation_key}."
                    )

                relation_keys.add(relation_key)

        return self

### Contrato raíz devuelto por la IA

In [ ]:
class BOEAIExtraction(ContractModel):
    """
    Salida generada por la IA para una publicación del BOE.

    boe_id y publication_date no se incluyen porque proceden de los metadatos
    fiables del pipeline y se incorporan posteriormente.
    """

    classification_status: ClassificationStatus

    document_scope: DocumentScope | None = None

    scope_reason: NonEmptyText = Field(
        description=(
            "Justificación breve y verificable de la clasificación documental."
        )
    )

    publication_events: list[PublicationEvent] = Field(
        default_factory=list
    )

    extraction_notes: NonEmptyText | None = Field(
        default=None,
        description=(
            "Advertencias sobre ambigüedades, contradicciones o información "
            "que no pudo atribuirse con suficiente seguridad."
        ),
    )

    @model_validator(mode="after")
    def validate_classification_and_scope(self) -> Self:
        """Comprueba la coherencia entre clasificación, alcance y eventos."""

        has_events = bool(self.publication_events)

        if self.classification_status == ClassificationStatus.UNCERTAIN:
            if self.document_scope is not None:
                raise ValueError(
                    "Una publicación incierta no debe tener document_scope."
                )

            if has_events:
                raise ValueError(
                    "Una publicación incierta no debe contener "
                    "publication_events."
                )

            return self

        if self.document_scope is None:
            raise ValueError(
                "Una publicación clasificada debe tener document_scope."
            )

        is_project_specific = (
            self.document_scope
            == DocumentScope.ENERGY_PROJECT_SPECIFIC
        )

        if is_project_specific and not has_events:
            raise ValueError(
                "Una publicación específica de proyecto debe contener "
                "al menos un PublicationEvent."
            )

        if not is_project_specific and has_events:
            raise ValueError(
                "Solo las publicaciones específicas de proyecto pueden "
                "contener PublicationEvent."
            )

        return self

### Contrato persistible enriquecido por el pipeline

In [ ]:
class BOEProjectExtraction(BOEAIExtraction):
    """
    Resultado completo que se persistirá.

    Añade al contenido generado por la IA los metadatos obtenidos directamente
    de la fuente BOE.
    """

    boe_id: BOEId
    publication_date: date


def build_boe_project_extraction(
    ai_extraction: BOEAIExtraction,
    *,
    boe_id: str,
    publication_date: date,
) -> BOEProjectExtraction:
    """
    Incorpora los metadatos fiables del BOE al resultado generado por la IA.

    La función evita pedir al modelo que reproduzca el identificador o que
    distinga la fecha de publicación de otras fechas presentes en el texto.
    """

    return BOEProjectExtraction(
        **ai_extraction.model_dump(),
        boe_id=boe_id,
        publication_date=publication_date,
    )

El objetivo final no es saber cuántas subestaciones, líneas o posiciones eléctricas hay, sino reconstruir para cada proyecto energético, un cronología como:
````
Proyecto X
- 2021-07-07: información pública AAP/AAC/DUP
- 2023-01-31: DIA favorable
- 2023-04-28: autorización administrativa previa
- 2024-...: autorización administrativa de construcción
- estado actual inferido: autorizado / en tramitación / archivado / denegado / etc.`
````

La solución óptima sería diseñar el contrato en torno a cuatro ideas:
- BOE publication: documento publicado.
- Administrative event: acto o conjunto de actos administrativos publicados.
- Project mention: proyecto energético afectado por ese acto.
- Associated infrastructure summary: resumen textual de infraestructura auxiliar, no tabla exhaustiva de activos.

In [18]:
INSTRUCTIONS = """
Eres un extractor canónico de información estructurada de documentos del BOE sobre proyectos energéticos.

Devuelve exclusivamente JSON válido conforme al esquema BOEProjectExtraction.
No incluyas explicaciones fuera del JSON.

Objetivo:
Extraer, desde una única publicación BOE, información suficiente para reconstruir posteriormente la evolución administrativa de proyectos energéticos mediante reglas deterministas.

Prioridad de reglas:
1. Cumple estrictamente el esquema BOEProjectExtraction.
2. Extrae solo información respaldada por el texto del BOE.
3. Usa los valores enum definidos por el contrato.
4. Si existe ambigüedad, conserva la evidencia y no fuerces una interpretación.

Unidad de extracción:
- BOEProjectExtraction representa una publicación BOE concreta.
- publication_events representa el hecho administrativo principal publicado en esa publicación.
- administrative_actions recoge los actos administrativos publicados.
- project_mentions recoge proyectos o instalaciones energéticas sustantivas afectadas.
- associated_infrastructure resume infraestructura auxiliar relevante, sin descomponerla exhaustivamente.
- No reconstruyas el ciclo de vida completo del proyecto.
- No agrupes publicaciones BOE.
- No generes identificadores globales.
- Usa local_project_id internos: project_1, project_2, project_3.

Reglas generales:
- Extrae únicamente información explícita del título o texto del BOE.
- No inventes, completes, corrijas ni resuelvas información con conocimiento externo.
- Si un dato opcional no aparece, usa null.
- Si una lista no tiene elementos, usa [].
- Usa "desconocido" solo en campos enum cuando la clasificación no pueda determinarse.
- No escribas "desconocido", "no consta" o "no aplica" en campos textuales salvo que aparezcan literalmente en el BOE.
- Toda información relevante debe tener evidence textual específica.
- Distingue el hecho publicado de los antecedentes históricos.
- Los antecedentes no deben crear publication_events ni administrative_actions salvo que formen parte del acto publicado actual.
- Los campos textuales deben conservarse en español: name, aliases, description, power_normalization_note, event_summary, relevance_reason, extraction_notes y evidence.

Relevancia:
- energy_relevance = "relevante" si el documento trata de un proyecto o infraestructura energética concreta.
- energy_relevance = "no_relevante" si es normativo, estadístico, tarifario, presupuestario, genérico o no vinculado a un proyecto concreto.
- energy_relevance = "dudoso" si contiene vocabulario energético pero no permite identificar un proyecto o infraestructura concreta.
- is_project_specific debe ser true solo si se identifica un proyecto, instalación o infraestructura energética concreta.
- relevance_reason debe justificar brevemente la clasificación.

PublicationEvent:
- Cada publicación BOE debe generar normalmente un único PublicationEvent.
- Crea varios PublicationEvent solo si hay hechos administrativos principales independientes sobre proyectos o instalaciones sustantivas distintas.
- No crees un PublicationEvent separado para cada trámite si esos trámites forman parte del mismo hecho administrativo.
- Si hay varios actos sobre el mismo hecho principal, inclúyelos en administrative_actions.
- event_summary debe ser breve, no nulo y en español.
- event_type debe describir la evolución material o administrativa principal: proyecto_nuevo, hibridacion, modificacion, repotenciacion, incorporacion_almacenamiento, proyecto_infraestructura_autonoma, cambio_titularidad, terminacion, otro o desconocido.

AdministrativeAction:
- administrative_actions debe recoger los actos administrativos publicados en el BOE.
- procedure_stage identifica el trámite o acto administrativo.
- procedure_decision identifica la decisión publicada.
- Extrae todos los actos administrativos que formen parte del objeto de publicación.
- No conviertas antecedentes históricos en administrative_actions del evento actual.
- Cada administrative_action debe tener evidence específica.

EnergyProjectMention:
- project_mentions debe recoger solo proyectos o instalaciones sustantivas afectadas por el hecho publicado.
- Extrae parques eólicos, plantas fotovoltaicas, almacenamiento, hibridaciones, repotenciaciones, modificaciones e infraestructuras autónomas cuando sean objeto principal.
- No extraigas líneas, SET, subestaciones, posiciones o evacuación auxiliar como EnergyProjectMention salvo que sean el objeto principal del BOE.
- Si una infraestructura es auxiliar del proyecto principal, resúmela en associated_infrastructure.
- No extraigas proyectos mencionados solo como contexto, antecedentes, agrupaciones, complejos energéticos, comparaciones o referencias informativas.
- Cada project_mention debe tener evidence específica.
- role_in_event debe indicar si la mención es objeto_principal, referencia_existente, referencia_asociada o desconocido.
- status_in_document solo refleja el estado mencionado o fuertemente implicado en esta publicación concreta, no el estado consolidado.

Technical attributes:
- technical_attributes recoge el tipo de instalación o tecnología y las   magnitudes técnicas explícitas.
- installation_type es una clasificación controlada, no una copia literal   del BOE.
- power_mw recoge la potencia de la instalación expresada en MW, independientemente de que corresponda a generación, almacenamiento, una línea, una subestación u otra instalación.
- peak_power_mwp recoge exclusivamente la potencia pico fotovoltaica expresada en MWp.
- storage_capacity_mwh recoge exclusivamente la capacidad energética del almacenamiento expresada en MWh.
- No transfieras potencias entre proyectos ni entre bloques técnicos.
- Cada valor de power_mw debe corresponder al installation_type del mismo bloque técnico.
- No deduzcas potencias salvo que la equivalencia sea explícita.
- Explica conversiones, cálculos o discrepancias en power_normalization_note.
- Si el BOE distingue potencia instalada, nominal, máxima, de acceso o de evacuación, extrae en power_mw únicamente la potencia que describa la instalación representada por ese bloque técnico. Conserva el matiz en description o power_normalization_note.

Interpretación numérica:
- Interpreta números con formato español.
- "31,172 MW" = 31.172 MW.
- "28.000 kW" = 28000 kW.
- Si el texto incluye número de equipos y potencia unitaria, calcula la potencia total solo si la equivalencia es explícita.
- Si hay discrepancia entre cálculo y cifra textual, consérvala en power_normalization_note sin afirmar que el BOE contiene una errata.

AssociatedInfrastructureSummary:
- associated_infrastructure resume infraestructura auxiliar relevante y no debe ser exhaustiva.
- Usa has_evacuation_infrastructure = true si se menciona evacuación, red colectora, línea de evacuación o infraestructura común.
- Usa has_electrical_substation = true si se mencionan SET, SE o subestaciones.
- Usa has_grid_connection = true si se menciona punto de conexión, REE, red de transporte/distribución o posición de conexión.
- Usa has_shared_infrastructure = true si la infraestructura parece compartida, colectora o común.
- description debe resumir la infraestructura en español.
- evidence debe justificar el resumen.

Hibridación y almacenamiento:
- Si el evento principal es una hibridación, event_type = "hibridacion".
- Extrae la nueva instalación como objeto_principal.
- Extrae la instalación existente como referencia_existente si aparece explícitamente.
- No crees relaciones entre activos: el contrato no contiene asset_relations.
- Si el evento principal es añadir almacenamiento, event_type = "incorporacion_almacenamiento".
- El almacenamiento puede ser objeto principal, no solo infraestructura auxiliar.

Participantes:
- Extrae promotores, copromotores, titulares, operadores, gestores de red, órganos sustantivos, órganos ambientales y administraciones solo si aparecen explícitamente.
- Conserva la denominación literal.
- No inventes CIF, NIF ni identificadores societarios.
- No resuelvas entidades por conocimiento externo.

Localizaciones:
- Extrae solo municipios, provincias y comunidades autónomas explícitamente mencionados.
- Conserva la forma textual del BOE.
- No normalices nombres ni generes códigos INE.
- Usa province_hint y autonomous_community_hint solo si aparecen explícitamente o en contexto inmediato.
- Si no hay localización explícita para un project_mention, devuelve locations = [].

Notas:
- extraction_notes debe registrar incertidumbres relevantes.
"""

In [19]:
MAPPING_HINTS = """
Mapeos orientativos BOE -> valores enum:

procedure_stage:
- solicitud, solicitud de autorización -> solicitud_tramitacion
- solicitud de evaluación ambiental, solicitud de determinación de afección ambiental -> solicitud_tramitacion_ambiental
- información pública, se somete a información pública -> informacion_publica
- evaluación de impacto ambiental -> evaluacion_impacto_ambiental
- declaración de impacto ambiental, DIA -> declaracion_impacto_ambiental
- informe de impacto ambiental -> informe_impacto_ambiental
- informe de determinación de afección ambiental, IDAA -> informe_determinacion_afeccion_ambiental
- autorización administrativa previa, AAP -> autorizacion_administrativa_previa
- autorización administrativa de construcción, AAC -> autorizacion_administrativa_construccion
- autorización de explotación -> autorizacion_explotacion
- declaración de utilidad pública, DUP -> declaracion_utilidad_publica
- relación de bienes y derechos afectados -> relacion_bienes_derechos_afectados
- levantamiento de actas previas a la ocupación -> levantamiento_actas_previas_ocupacion
- actas de ocupación -> actas_ocupacion
- modificación -> modificacion
- prórroga -> prorroga
- archivo del expediente -> archivo_expediente
- desistimiento -> desistimiento
- inadmisión -> inadmision
- cambio de titularidad, transmisión de titularidad -> cambio_titularidad

procedure_decision:
- solicita, solicitud -> solicitado
- subsanación -> subsanado
- se formula -> formulado
- favorable -> favorable
- desfavorable -> desfavorable
- ambientalmente viable -> ambientalmente_viable
- ambientalmente no viable -> ambientalmente_no_viable
- debe someterse a evaluación ambiental adicional u ordinaria -> requiere_evaluacion_ambiental_adicional
- no debe someterse a evaluación ambiental adicional u ordinaria -> no_requiere_evaluacion_ambiental_adicional
- se somete a información pública -> sometido_informacion_publica
- se convoca -> convocado
- se declara de utilidad pública -> declarado_utilidad_publica
- se aprueba -> aprobado
- se autoriza, se otorga autorización -> autorizado
- se deniega -> denegado
- se archiva -> archivado
- desiste -> desistido
- se inadmite -> inadmitido

event_type:
- nueva planta, nuevo parque, nueva instalación -> proyecto_nuevo
- hibridación -> hibridacion
- modificación, ampliación, cambio de configuración -> modificacion
- repotenciación -> repotenciacion
- almacenamiento asociado, incorporación de baterías, BESS asociado -> incorporacion_almacenamiento
- línea, subestación o infraestructura autónoma como objeto principal -> proyecto_infraestructura_autonoma
- cambio/transmisión de titularidad -> cambio_titularidad
- archivo, desistimiento, inadmisión o denegación final -> terminacion

installation_type:
- planta solar, planta fotovoltaica, PFV, FV -> fotovoltaica
- parque eólico, PE, aerogeneradores -> eolica
- termosolar, solar termoeléctrica -> termosolar
- hidroeléctrica -> hidroelectrica
- biomasa -> biomasa
- biogás -> biogas
- hidrógeno verde, hidrógeno renovable -> hidrogeno_verde
- BESS, baterías, almacenamiento -> almacenamiento
- línea eléctrica, línea aérea, línea subterránea, LAT, LSAT -> linea_electrica
- subestación, SET, SE -> subestacion_electrica
- evacuación, infraestructura de evacuación, infraestructura común de evacuación -> infraestructura_evacuacion
"""

In [20]:
### Agent instructions

AGENT_INSTRUCTIONS = "\n\n".join(
    [
        INSTRUCTIONS.strip(),
        MAPPING_HINTS.strip(),
    ]
)

### Build Agent

In [21]:
def build_agent(
    model,
    output_type: type[BaseModel],
    instructions: str,
    *,
    retries: int = 3,
) -> Agent:
    """
    Build a Pydantic AI agent for structured BOE extraction.

    Parameters
    ----------
    model
        Model identifier or model instance accepted by Pydantic AI.
    output_type
        Pydantic model defining the expected structured output.
    instructions
        Full extraction instructions passed to the model.
    retries
        Number of retries for structured output validation.

    Returns
    -------
    Agent
        Configured Pydantic AI agent.
    """
    return Agent(
        model,
        output_type=output_type,
        instructions=instructions,
        retries=retries,
    )

In [22]:
# Ollama model builder

from pydantic_ai.models.ollama import OllamaModel
from pydantic_ai.providers.ollama import OllamaProvider


def build_ollama_model(
    model_name: str,
    *,
    base_url: str = "http://localhost:11434/v1",
) -> OllamaModel:
    """
    Build an Ollama model instance compatible with Pydantic AI.

    Parameters
    ----------
    model_name
        Local Ollama model name, for example "qwen3:8b".
    base_url
        Ollama OpenAI-compatible endpoint.

    Returns
    -------
    OllamaModel
        Configured Ollama model.
    """
    return OllamaModel(
        model_name,
        provider=OllamaProvider(base_url=base_url),
    )

### Modelos disponibles

In [23]:
MODEL_PROVIDER = "gemini"
# MODEL_PROVIDER = "ollama"


if MODEL_PROVIDER == "gemini":
    AI_MODEL_NAME = "google:gemini-2.5-flash"
    AI_MODEL = AI_MODEL_NAME
    AGENT_RETRIES = 3

elif MODEL_PROVIDER == "ollama":
    AI_MODEL_NAME = "qwen3:8b"
    AI_MODEL = build_ollama_model(AI_MODEL_NAME)
    AGENT_RETRIES = 4  # Local models usually need slightly more validation retries.

else:
    raise ValueError(
        f"Unsupported model provider: {MODEL_PROVIDER}"
    )

In [24]:
agent = build_agent(
    model=AI_MODEL,
    output_type=BOEProjectExtraction,
    instructions=AGENT_INSTRUCTIONS,
    retries=AGENT_RETRIES,
)

In [25]:
TEXT_LIMIT = 4000

AI_EXTRACTION_LOG_COLUMNS = [
    "identificador_boe",
    "boe_id_extracted",
    "fecha_publicacion",
    "publication_date_extracted",
    "titulo",
    "energy_relevance",
    "is_project_specific",
    "n_publication_events",
    "extraction_json",
    "extracted_at",
    "model_name",
    "extraction_status",
    "error_type",
    "parse_error",
]

### Funciones

#### Utilidades internas

In [26]:
def empty_ai_extractions_log() -> pd.DataFrame:
    """
    Crea un log vacío de extracciones IA.

    Se usa cuando todavía no existe el fichero Parquet acumulado de
    extracciones. Devuelve un DataFrame con las columnas esperadas, pero sin
    registros.
    """
    return pd.DataFrame(columns=AI_EXTRACTION_LOG_COLUMNS)

In [27]:
def normalise_ai_extractions_log(
    ai_extractions: pd.DataFrame,
) -> pd.DataFrame:
    """
    Normaliza la estructura tabular del log de extracciones IA.

    Garantiza que existan todas las columnas definidas en
    `AI_EXTRACTION_LOG_COLUMNS`, conserva posibles columnas adicionales al final
    y normaliza los tipos mínimos necesarios para trabajar de forma estable.

    Esta función no valida el contenido semántico del JSON extraído. Solo
    prepara el DataFrame del log.
    """
    ai_extractions = ai_extractions.copy()

    for col in AI_EXTRACTION_LOG_COLUMNS:
        if col not in ai_extractions.columns:
            ai_extractions[col] = pd.NA

    extra_cols = [
        col
        for col in ai_extractions.columns
        if col not in AI_EXTRACTION_LOG_COLUMNS
    ]

    ai_extractions = ai_extractions[
        AI_EXTRACTION_LOG_COLUMNS + extra_cols
    ]

    if ai_extractions.empty:
        return ai_extractions

    ai_extractions["identificador_boe"] = (
        ai_extractions["identificador_boe"]
        .astype("string")
    )

    ai_extractions["boe_id_extracted"] = (
        ai_extractions["boe_id_extracted"]
        .astype("string")
    )

    ai_extractions["fecha_publicacion"] = pd.to_datetime(
        ai_extractions["fecha_publicacion"],
        errors="coerce",
    )

    ai_extractions["publication_date_extracted"] = pd.to_datetime(
        ai_extractions["publication_date_extracted"],
        errors="coerce",
    )

    return ai_extractions


In [28]:
def validate_extraction_metadata(
    row: pd.Series,
    extraction: BOEProjectExtraction,
) -> None:
    """
    Valida la coherencia entre el documento fuente y la extracción IA.

    El identificador BOE y la fecha de publicación proceden del dataset fuente
    y se consideran metadatos canónicos. La IA puede devolver esos mismos
    campos dentro del JSON, pero no debe contradecirlos.

    La validación falla si:
    - `extraction.boe_id` no coincide con `row["identificador"]`.
    - `extraction.publication_date` y `row["fecha_publicacion"]` existen y son
      fechas distintas.

    No falla si una de las dos fechas es nula, porque en ese caso no hay
    contradicción verificable.
    """
    source_boe_id = safe_str(row["identificador"])
    extracted_boe_id = safe_str(extraction.boe_id)

    if source_boe_id != extracted_boe_id:
        raise ValueError(
            "Identificador BOE incoherente: "
            f"fuente={source_boe_id!r}, "
            f"extraccion={extracted_boe_id!r}"
        )

    source_date = to_date_or_none(row["fecha_publicacion"])
    extracted_date = to_date_or_none(extraction.publication_date)

    if (
        source_date is not None
        and extracted_date is not None
        and source_date != extracted_date
    ):
        raise ValueError(
            "Fecha de publicación incoherente: "
            f"fuente={source_date}, "
            f"extraccion={extracted_date}"
        )

#### Construcción de prompt

In [29]:
def build_prompt(
    row: pd.Series,
    text_limit: int = TEXT_LIMIT,
) -> str:
    """
    Construye el prompt documental enviado al agente de extracción.
    """
    texto_limpio = safe_str(row["texto_limpio"])

    return f"""
Identificador BOE: {row["identificador"]}
Fecha publicación: {row["fecha_publicacion"]}
Título: {row["titulo"]}

Texto:
{texto_limpio[:text_limit]}
""".strip()

#### Registro de extracción

In [30]:
def build_ai_extraction_record(
    row: pd.Series,
    extraction: BOEProjectExtraction,
    model_name: str,
) -> dict:
    """
    Construye un registro tabular a partir de una extracción IA validada.

    La clave documental canónica se toma del dataset fuente. Los valores
    devueltos por la IA se conservan también para auditoría.
    """
    validate_extraction_metadata(
        row=row,
        extraction=extraction,
    )

    return {
        "identificador_boe": row["identificador"],
        "boe_id_extracted": extraction.boe_id,
        "fecha_publicacion": pd.to_datetime(
            row["fecha_publicacion"],
            errors="coerce",
        ),
        "publication_date_extracted": pd.to_datetime(
            extraction.publication_date,
            errors="coerce",
        ),
        "titulo": row["titulo"],
        "energy_relevance": enum_value(extraction.energy_relevance),
        "is_project_specific": extraction.is_project_specific,
        "n_publication_events": len(extraction.publication_events),
        "extraction_json": extraction.model_dump_json(),
        "extracted_at": datetime.now(timezone.utc).isoformat(),
        "model_name": model_name,
        "extraction_status": "ok",
        "error_type": None,
        "parse_error": None,
    }

In [31]:
def build_ai_error_record(
    row: pd.Series,
    model_name: str,
    error: Exception,
) -> dict:
    """
    Construye un registro tabular cuando falla la extracción IA.
    """
    return {
        "identificador_boe": row["identificador"],
        "boe_id_extracted": None,
        "fecha_publicacion": pd.to_datetime(
            row["fecha_publicacion"],
            errors="coerce",
        ),
        "publication_date_extracted": pd.NaT,
        "titulo": row["titulo"],
        "energy_relevance": None,
        "is_project_specific": None,
        "n_publication_events": None,
        "extraction_json": None,
        "extracted_at": datetime.now(timezone.utc).isoformat(),
        "model_name": model_name,
        "extraction_status": "error",
        "error_type": type(error).__name__,
        "parse_error": str(error),
    }

#### Carga y upsert incremental

In [32]:
def load_ai_extractions(
    output_path: Path,
) -> pd.DataFrame:
    """
    Carga el log acumulado de extracciones IA.

    Si el fichero no existe, devuelve un DataFrame vacío con las columnas
    esperadas.
    """
    if not output_path.exists():
        return empty_ai_extractions_log()

    ai_extractions = pd.read_parquet(output_path)

    return normalise_ai_extractions_log(ai_extractions)

In [33]:
def upsert_ai_extractions(
    new_ai_extractions: pd.DataFrame,
    output_path: Path,
) -> pd.DataFrame:
    """
    Inserta o actualiza extracciones IA en un Parquet acumulado.

    Conserva la última extracción de cada `identificador_boe`.

    Esta función debe usarse solo para la tabla fuente de extracciones IA. Las
    tablas derivadas deben regenerarse desde esta tabla.
    """
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    existing_ai_extractions = load_ai_extractions(output_path)

    new_ai_extractions = normalise_ai_extractions_log(
        new_ai_extractions
    )

    if new_ai_extractions.empty:
        return existing_ai_extractions

    ai_extractions = pd.concat(
        [
            existing_ai_extractions,
            new_ai_extractions,
        ],
        ignore_index=True,
    )

    ai_extractions = ai_extractions.drop_duplicates(
        subset=["identificador_boe"],
        keep="last",
    )

    ai_extractions = normalise_ai_extractions_log(
        ai_extractions
    )

    save_parquet(
        ai_extractions,
        output_path,
    )

    return ai_extractions

### Cargar candidatos BOE

In [34]:
df = pd.read_parquet(BOE_CANDIDATES_DOCS_TEXT_PATH)

required_input_cols = {
    "identificador",
    "fecha_publicacion",
    "titulo",
    "xml_status",
    "texto_limpio",
}

validate_required_columns(
    df,
    required_input_cols,
)

df = df.loc[
    (df["xml_status"] == "ok")
    & df["texto_limpio"].notna()
    & (df["texto_limpio"].astype(str).str.len() > 0)
].copy()

df["identificador"] = df["identificador"].astype("string")

print(f"{len(df)=}")

len(df)=1266


### Filtrar BOEs ya procesados correctamente

In [35]:
ai_extractions = load_ai_extractions(
    BOE_AI_EXTRACTIONS_PATH
)

processed_ids = set(
    ai_extractions.loc[
        ai_extractions["extraction_status"] == "ok",
        "identificador_boe",
    ]
    .dropna()
    .astype(str)
)

pending_df = df.loc[
    ~df["identificador"].isin(processed_ids)
].copy()

print(f"{len(processed_ids)=}")
print(f"{len(pending_df)=}")

len(processed_ids)=0
len(pending_df)=1266


### Seleccionar proyectos de test

#### PE Badulaque

In [36]:
target_ids_badulaque = [
    "BOE-B-2021-32560",
    "BOE-A-2023-2598",
    "BOE-A-2023-10306",
    "BOE-B-2023-19082",
    "BOE-A-2024-16664",
]

#### FV Andévalo

Buenos casos de prueba para comprobar que el sistema no agrupa proyectos distintos simplemente porque comparten infraestructura de evacuación o mencionan FV Andévalo.

In [37]:
target_ids_andevalo = [
    # Proyecto FV Andévalo e hibridaciones
    "BOE-B-2024-26379",
    "BOE-A-2025-18285",
    "BOE-B-2026-3596",

    # Antecedentes y referencias indirectas
    "BOE-A-2022-24404",  # FV Majal Alto
    "BOE-A-2024-9608",   # FV La Puebla 1
    "BOE-A-2025-26110",  # FV La Puebla 1
    "BOE-A-2026-7629",   # FV La Puebla 3 y 4
    "BOE-A-2026-13454",  # FV La Puebla 3
]

In [38]:
target_ids = target_ids_badulaque + target_ids_andevalo

In [39]:
missing_target_ids = sorted(
    set(target_ids) - set(df["identificador"].astype(str))
)

if missing_target_ids:
    print("target_ids no encontrados en `df`:")
    print(missing_target_ids)

df_test_initial = df.loc[
    df["identificador"].isin(target_ids)
].copy()

df_test_pending = df_test_initial.loc[
    ~df_test_initial["identificador"].isin(processed_ids)
].copy()

print(f"{len(df_test_initial)=}")
print(f"{len(df_test_pending)=}")

len(df_test_initial)=13
len(df_test_pending)=13


In [40]:
display(
    df_test_initial[
        [
            "identificador",
            "fecha_publicacion",
            "titulo",
        ]
    ].sort_values("identificador")
)

,identificador,fecha_publicacion,titulo
16,BOE-A-2022-24404,2022-12-30,"Resolución de 22 de diciembre de 2022, de la D..."
98,BOE-A-2023-10306,2023-04-28,"Resolución de 17 de abril de 2023, de la Direc..."
76,BOE-A-2023-2598,2023-01-31,"Resolución de 23 de enero de 2023, de la Direc..."
213,BOE-A-2024-16664,2024-08-10,"Resolución de 22 de julio de 2024, de la Direc..."
142,BOE-A-2024-9608,2024-05-13,"Resolución de 6 de mayo de 2024, de la Direcci..."
235,BOE-A-2025-18285,2025-09-15,"Resolución de 7 de agosto de 2025, de la Direc..."
275,BOE-A-2025-26110,2025-12-19,"Resolución de 17 de noviembre de 2025, de la D..."
1264,BOE-A-2026-13454,2026-06-20,"Resolución de 3 de junio de 2026, de la Direcc..."
310,BOE-A-2026-7629,2026-04-03,"Resolución de 17 de marzo de 2026, de la Direc..."
6,BOE-B-2021-32560,2021-07-07,Anuncio del Área de Industria y Energía de la ...


In [41]:
## 5. Extracción con agente sobre los pendientes

# Modo de ejecución.
# Durante el desarrollo se usan solo los casos de prueba pendientes
# para validar el contrato y revisar errores de forma controlada.
run_df = df_test_pending.copy()

# TODO: Para procesar todos los BOE pendientes, sustituir la línea anterior por:
# run_df = pending_df.copy()


# Lista donde se acumula un registro tabular por cada documento procesado.
# Cada registro será una fila del log de extracciones IA.
records = []


# Variables auxiliares de depuración.
# No se guardan en parquet. Solo permiten inspeccionar en el notebook
# el último prompt, resultado, extracción o error producido.
last_prompt = None
last_result = None
last_extraction = None
last_error = None


print(f"{len(run_df)=}")


for i, (_, row) in enumerate(run_df.iterrows(), start=1):
    boe_id = row["identificador"]

    print(f"[{i}/{len(run_df)}] Extrayendo {boe_id}")

    # Construye el prompt documental a partir de la fila BOE.
    # Incluye identificador, fecha, título y texto limpio recortado.
    last_prompt = build_prompt(row)

    try:
        # Ejecuta el agente IA y valida la salida contra BOEProjectExtraction.
        last_result = await agent.run(last_prompt)
        last_extraction = last_result.output
        last_error = None

        # Convierte la extracción Pydantic validada en una fila tabular.
        # Aquí también se valida que boe_id y publication_date no contradigan
        # los metadatos canónicos del documento fuente.
        record = build_ai_extraction_record(
            row=row,
            extraction=last_extraction,
            model_name=AI_MODEL_NAME,
        )

    except Exception as exc:
        # Si falla la llamada al modelo, la validación Pydantic o la validación
        # de metadatos, se registra una fila de error en lugar de interrumpir
        # todo el procesamiento.
        last_error = exc

        record = build_ai_error_record(
            row=row,
            model_name=AI_MODEL_NAME,
            error=exc,
        )

        print(f"  ERROR {type(exc).__name__}: {exc}")

    records.append(record)


# Convierte los registros nuevos en DataFrame.
new_ai_extractions = pd.DataFrame(records)

# Normaliza columnas y tipos mínimos del log antes de mostrar o guardar.
# Esto garantiza que las extracciones correctas y los errores tengan
# la misma estructura tabular.
new_ai_extractions = normalise_ai_extractions_log(
    new_ai_extractions
)

display(new_ai_extractions)

len(run_df)=13
[1/13] Extrayendo BOE-B-2021-32560
[2/13] Extrayendo BOE-A-2022-24404
[3/13] Extrayendo BOE-A-2023-2598
[4/13] Extrayendo BOE-A-2023-10306
[5/13] Extrayendo BOE-B-2023-19082
[6/13] Extrayendo BOE-A-2024-9608
[7/13] Extrayendo BOE-B-2024-26379
[8/13] Extrayendo BOE-A-2024-16664
[9/13] Extrayendo BOE-A-2025-18285
[10/13] Extrayendo BOE-A-2025-26110
[11/13] Extrayendo BOE-B-2026-3596
[12/13] Extrayendo BOE-A-2026-7629
[13/13] Extrayendo BOE-A-2026-13454


,identificador_boe,boe_id_extracted,fecha_publicacion,publication_date_extracted,titulo,energy_relevance,is_project_specific,n_publication_events,extraction_json,extracted_at,model_name,extraction_status,error_type,parse_error
0,BOE-B-2021-32560,BOE-B-2021-32560,2021-07-07,2021-07-07,Anuncio del Área de Industria y Energía de la ...,relevante,True,1,"{""boe_id"":""BOE-B-2021-32560"",""publication_date...",2026-07-10T16:05:08.847632+00:00,google:gemini-2.5-flash,ok,None,None
1,BOE-A-2022-24404,BOE-A-2022-24404,2022-12-30,2022-12-30,"Resolución de 22 de diciembre de 2022, de la D...",relevante,True,1,"{""boe_id"":""BOE-A-2022-24404"",""publication_date...",2026-07-10T16:05:22.466872+00:00,google:gemini-2.5-flash,ok,None,None
2,BOE-A-2023-2598,BOE-A-2023-2598,2023-01-31,2023-01-31,"Resolución de 23 de enero de 2023, de la Direc...",relevante,True,1,"{""boe_id"":""BOE-A-2023-2598"",""publication_date""...",2026-07-10T16:05:35.259029+00:00,google:gemini-2.5-flash,ok,None,None
3,BOE-A-2023-10306,BOE-A-2023-10306,2023-04-28,2023-04-28,"Resolución de 17 de abril de 2023, de la Direc...",relevante,True,1,"{""boe_id"":""BOE-A-2023-10306"",""publication_date...",2026-07-10T16:05:57.381621+00:00,google:gemini-2.5-flash,ok,None,None
4,BOE-B-2023-19082,BOE-B-2023-19082,2023-06-22,2023-06-22,Anuncio del Área Funcional de Industria y Ener...,relevante,True,1,"{""boe_id"":""BOE-B-2023-19082"",""publication_date...",2026-07-10T16:06:09.859613+00:00,google:gemini-2.5-flash,ok,None,None
5,BOE-A-2024-9608,BOE-A-2024-9608,2024-05-13,2024-05-13,"Resolución de 6 de mayo de 2024, de la Direcci...",relevante,True,1,"{""boe_id"":""BOE-A-2024-9608"",""publication_date""...",2026-07-10T16:06:28.612261+00:00,google:gemini-2.5-flash,ok,None,None
6,BOE-B-2024-26379,BOE-B-2024-26379,2024-07-13,2024-07-13,Anuncio del Área de Industria y Energía de la ...,relevante,True,1,"{""boe_id"":""BOE-B-2024-26379"",""publication_date...",2026-07-10T16:06:40.848907+00:00,google:gemini-2.5-flash,ok,None,None
7,BOE-A-2024-16664,BOE-A-2024-16664,2024-08-10,2024-08-10,"Resolución de 22 de julio de 2024, de la Direc...",relevante,True,1,"{""boe_id"":""BOE-A-2024-16664"",""publication_date...",2026-07-10T16:06:53.425715+00:00,google:gemini-2.5-flash,ok,None,None
8,BOE-A-2025-18285,BOE-A-2025-18285,2025-09-15,2025-09-15,"Resolución de 7 de agosto de 2025, de la Direc...",relevante,True,1,"{""boe_id"":""BOE-A-2025-18285"",""publication_date...",2026-07-10T16:07:13.410899+00:00,google:gemini-2.5-flash,ok,None,None
9,BOE-A-2025-26110,BOE-A-2025-26110,2025-12-19,2025-12-19,"Resolución de 17 de noviembre de 2025, de la D...",relevante,True,1,"{""boe_id"":""BOE-A-2025-26110"",""publication_date...",2026-07-10T16:08:02.173224+00:00,google:gemini-2.5-flash,ok,None,None


In [42]:
# Inserta o actualiza el log acumulado de extracciones IA.
# Si un identificador BOE ya existía, se conserva la última versión.
ai_extractions = upsert_ai_extractions(
    new_ai_extractions=new_ai_extractions,
    output_path=BOE_AI_EXTRACTIONS_PATH,
)

display(ai_extractions)

/tmp/ipykernel_746311/2444363489.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  ai_extractions = pd.concat(


,identificador_boe,boe_id_extracted,fecha_publicacion,publication_date_extracted,titulo,energy_relevance,is_project_specific,n_publication_events,extraction_json,extracted_at,model_name,extraction_status,error_type,parse_error
0,BOE-B-2021-32560,BOE-B-2021-32560,2021-07-07,2021-07-07,Anuncio del Área de Industria y Energía de la ...,relevante,True,1,"{""boe_id"":""BOE-B-2021-32560"",""publication_date...",2026-07-10T16:05:08.847632+00:00,google:gemini-2.5-flash,ok,None,None
1,BOE-A-2022-24404,BOE-A-2022-24404,2022-12-30,2022-12-30,"Resolución de 22 de diciembre de 2022, de la D...",relevante,True,1,"{""boe_id"":""BOE-A-2022-24404"",""publication_date...",2026-07-10T16:05:22.466872+00:00,google:gemini-2.5-flash,ok,None,None
2,BOE-A-2023-2598,BOE-A-2023-2598,2023-01-31,2023-01-31,"Resolución de 23 de enero de 2023, de la Direc...",relevante,True,1,"{""boe_id"":""BOE-A-2023-2598"",""publication_date""...",2026-07-10T16:05:35.259029+00:00,google:gemini-2.5-flash,ok,None,None
3,BOE-A-2023-10306,BOE-A-2023-10306,2023-04-28,2023-04-28,"Resolución de 17 de abril de 2023, de la Direc...",relevante,True,1,"{""boe_id"":""BOE-A-2023-10306"",""publication_date...",2026-07-10T16:05:57.381621+00:00,google:gemini-2.5-flash,ok,None,None
4,BOE-B-2023-19082,BOE-B-2023-19082,2023-06-22,2023-06-22,Anuncio del Área Funcional de Industria y Ener...,relevante,True,1,"{""boe_id"":""BOE-B-2023-19082"",""publication_date...",2026-07-10T16:06:09.859613+00:00,google:gemini-2.5-flash,ok,None,None
5,BOE-A-2024-9608,BOE-A-2024-9608,2024-05-13,2024-05-13,"Resolución de 6 de mayo de 2024, de la Direcci...",relevante,True,1,"{""boe_id"":""BOE-A-2024-9608"",""publication_date""...",2026-07-10T16:06:28.612261+00:00,google:gemini-2.5-flash,ok,None,None
6,BOE-B-2024-26379,BOE-B-2024-26379,2024-07-13,2024-07-13,Anuncio del Área de Industria y Energía de la ...,relevante,True,1,"{""boe_id"":""BOE-B-2024-26379"",""publication_date...",2026-07-10T16:06:40.848907+00:00,google:gemini-2.5-flash,ok,None,None
7,BOE-A-2024-16664,BOE-A-2024-16664,2024-08-10,2024-08-10,"Resolución de 22 de julio de 2024, de la Direc...",relevante,True,1,"{""boe_id"":""BOE-A-2024-16664"",""publication_date...",2026-07-10T16:06:53.425715+00:00,google:gemini-2.5-flash,ok,None,None
8,BOE-A-2025-18285,BOE-A-2025-18285,2025-09-15,2025-09-15,"Resolución de 7 de agosto de 2025, de la Direc...",relevante,True,1,"{""boe_id"":""BOE-A-2025-18285"",""publication_date...",2026-07-10T16:07:13.410899+00:00,google:gemini-2.5-flash,ok,None,None
9,BOE-A-2025-26110,BOE-A-2025-26110,2025-12-19,2025-12-19,"Resolución de 17 de noviembre de 2025, de la D...",relevante,True,1,"{""boe_id"":""BOE-A-2025-26110"",""publication_date...",2026-07-10T16:08:02.173224+00:00,google:gemini-2.5-flash,ok,None,None


In [43]:
# Resumen global del estado del log acumulado.
# Permite comprobar rápidamente cuántas extracciones han terminado en "ok"
# y cuántas han quedado como "error".
if not ai_extractions.empty:
    display(
        ai_extractions["extraction_status"]
        .value_counts(dropna=False)
        .rename_axis("extraction_status")
        .reset_index(name="n")
    )

,extraction_status,n
0,ok,13


In [44]:
# Vista compacta de los registros generados en esta ejecución.
# Sirve para revisar sin abrir el JSON completo:
# - relevancia energética,
# - si el documento es específico de proyecto,
# - número de eventos extraídos,
# - estado de extracción,
# - error, si lo hubo.
if not new_ai_extractions.empty:
    display(
        new_ai_extractions[
            [
                "identificador_boe",
                "fecha_publicacion",
                "energy_relevance",
                "is_project_specific",
                "n_publication_events",
                "extraction_status",
                "error_type",
                "parse_error",
            ]
        ]
    )

,identificador_boe,fecha_publicacion,energy_relevance,is_project_specific,n_publication_events,extraction_status,error_type,parse_error
0,BOE-B-2021-32560,2021-07-07,relevante,True,1,ok,None,None
1,BOE-A-2022-24404,2022-12-30,relevante,True,1,ok,None,None
2,BOE-A-2023-2598,2023-01-31,relevante,True,1,ok,None,None
3,BOE-A-2023-10306,2023-04-28,relevante,True,1,ok,None,None
4,BOE-B-2023-19082,2023-06-22,relevante,True,1,ok,None,None
5,BOE-A-2024-9608,2024-05-13,relevante,True,1,ok,None,None
6,BOE-B-2024-26379,2024-07-13,relevante,True,1,ok,None,None
7,BOE-A-2024-16664,2024-08-10,relevante,True,1,ok,None,None
8,BOE-A-2025-18285,2025-09-15,relevante,True,1,ok,None,None
9,BOE-A-2025-26110,2025-12-19,relevante,True,1,ok,None,None


In [45]:
extracted = load_ai_extractions(
    BOE_AI_EXTRACTIONS_PATH
)

In [46]:
def show_extraction_json(
    ai_extractions: pd.DataFrame,
    boe_id: str,
) -> None:
    """
    Muestra el JSON extraído para un identificador BOE.

    Usa la última extracción disponible para ese identificador y comprueba que
    esté en estado `ok`.
    """
    mask = ai_extractions["identificador_boe"].eq(boe_id)

    if not mask.any():
        raise ValueError(
            f"No hay extracción registrada para {boe_id}"
        )

    rows = ai_extractions.loc[mask].copy()

    rows["extracted_at"] = pd.to_datetime(
        rows["extracted_at"],
        errors="coerce",
    )

    row = (
        rows.sort_values("extracted_at")
        .iloc[-1]
    )

    if row["extraction_status"] != "ok":
        raise ValueError(
            f"La extracción de {boe_id} no está en estado ok. "
            f"Estado: {row['extraction_status']}. "
            f"Error: {row['parse_error']}"
        )

    if pd.isna(row["extraction_json"]):
        raise ValueError(
            f"La extracción de {boe_id} no contiene JSON."
        )

    try:
        parsed_json = json.loads(row["extraction_json"])
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"El JSON extraído para {boe_id} no es válido."
        ) from exc

    print(
        json.dumps(
            parsed_json,
            indent=2,
            ensure_ascii=False,
        )
    )

In [47]:
boe_id_to_check = "BOE-B-2021-32560"

show_extraction_json(
    extracted,
    boe_id_to_check,
)

{
  "boe_id": "BOE-B-2021-32560",
  "publication_date": "2021-07-07",
  "energy_relevance": "relevante",
  "is_project_specific": true,
  "relevance_reason": "Anuncio de sometimiento a información pública del Estudio de Impacto Ambiental y la solicitud de Autorización Administrativa Previa de un parque eólico y su infraestructura de evacuación.",
  "publication_events": [
    {
      "event_type": "proyecto_nuevo",
      "administrative_actions": [
        {
          "procedure_stage": "informacion_publica",
          "procedure_decision": "sometido_informacion_publica",
          "evidence": "se somete al trámite de información pública, de forma conjunta, el Estudio de Impacto Ambiental y la solicitud de Autorización Administrativa Previa"
        },
        {
          "procedure_stage": "solicitud_tramitacion_ambiental",
          "procedure_decision": "solicitado",
          "evidence": "solicitud de Autorización Administrativa Previa del Parque Eólico Badulaque de 90 MW y su infr

## 1. Flattening data in field `extraction_json`

### Funciones

#### Columnas esperadas

In [48]:
# Las columnas esperadas sirven para que cada función devuelva siempre un DataFrame con la misma estructura, incluso cuando no hay registros.
# El objetivo es tener un pipeline más determinista.

PUBLICATION_EVENTS_COLUMNS = [
    "event_id",
    "identificador_boe",
    "fecha_publicacion",
    "event_index",
    "event_type",
    "event_summary",
    "evidence",
]

ADMINISTRATIVE_ACTIONS_COLUMNS = [
    "action_id",
    "event_id",
    "identificador_boe",
    "fecha_publicacion",
    "action_index",
    "procedure_stage",
    "procedure_decision",
    "evidence",
]

PROJECT_MENTIONS_COLUMNS = [
    "project_mention_id",
    "event_id",
    "identificador_boe",
    "fecha_publicacion",
    "local_project_id",
    "project_name",
    "project_name_norm",
    "role_in_event",
    "status_in_document",
    "case_file_reference",
    "evidence",
]

PROJECT_TECHNICAL_ATTRIBUTES_COLUMNS = [
    "project_technical_attribute_id",
    "project_mention_id",
    "event_id",
    "identificador_boe",
    "fecha_publicacion",
    "attribute_index",
    "installation_type",
    "power_mw",
    "peak_power_mwp",
    "storage_capacity_mwh",
    "description",
    "power_normalization_note",
    "evidence",
]

PROJECT_PARTICIPANTS_COLUMNS = [
    "project_participant_id",
    "project_mention_id",
    "event_id",
    "identificador_boe",
    "fecha_publicacion",
    "participant_index",
    "participant_name",
    "participant_name_norm",
    "participant_role",
    "evidence",
]

PROJECT_LOCATIONS_COLUMNS = [
    "project_location_id",
    "project_mention_id",
    "event_id",
    "identificador_boe",
    "fecha_publicacion",
    "location_index",
    "municipality_raw",
    "municipality_raw_norm",
    "province_hint_raw",
    "province_hint_raw_norm",
    "autonomous_community_hint_raw",
    "autonomous_community_hint_raw_norm",
    "location_evidence",
]

PROJECT_ALIASES_COLUMNS = [
    "project_alias_id",
    "project_mention_id",
    "event_id",
    "identificador_boe",
    "fecha_publicacion",
    "alias_index",
    "alias",
    "alias_norm",
]

ASSOCIATED_INFRASTRUCTURE_COLUMNS = [
    "associated_infrastructure_id",
    "event_id",
    "identificador_boe",
    "fecha_publicacion",
    "has_evacuation_infrastructure",
    "has_electrical_substation",
    "has_grid_connection",
    "has_shared_infrastructure",
    "description",
    "evidence",
]

#### Utilidades de flattening

In [49]:
INVALID_LOCATION_VALUES = {
    "",
    "no consta",
    "desconocido",
    "no aplica",
    "ninguno",
}


def empty_df(columns: list[str]) -> pd.DataFrame:
    """
    Devuelve un DataFrame vacío con columnas estables.

    Evita que una tabla derivada quede sin columnas cuando no hay registros.
    """
    return pd.DataFrame(columns=columns)


def iter_valid_extractions(
    ai_extractions: pd.DataFrame,
):
    """
    Itera sobre extracciones IA válidas.

    Solo procesa filas con `extraction_status == "ok"` y `extraction_json`
    no nulo. Cada JSON se valida de nuevo contra `BOEProjectExtraction`.
    """
    required_cols = {
        "extraction_status",
        "extraction_json",
    }

    validate_required_columns(
        ai_extractions,
        required_cols,
    )

    valid_rows = ai_extractions.loc[
        (ai_extractions["extraction_status"] == "ok")
        & ai_extractions["extraction_json"].notna()
    ]

    for _, row in valid_rows.iterrows():
        yield BOEProjectExtraction.model_validate_json(
            row["extraction_json"]
        )


def make_event_id(
    boe_id: str,
    event_idx: int,
) -> str:
    """
    Construye un identificador estable de evento publicado.
    """
    return f"{boe_id}_event_{event_idx}"


def make_project_mention_id(
    event_id: str,
    local_project_id: str,
) -> str:
    """
    Construye un identificador estable de mención de proyecto.
    """
    return f"{event_id}_{local_project_id}"

#### flatten_publication_events

In [50]:
def flatten_publication_events(
    ai_extractions: pd.DataFrame,
) -> pd.DataFrame:
    """
    Aplana `publication_events`.

    Una fila representa un evento administrativo publicado en un documento BOE.
    No representa el ciclo de vida consolidado del proyecto.
    """
    records = []

    for extraction in iter_valid_extractions(ai_extractions):
        for event_idx, event in enumerate(
            extraction.publication_events,
            start=1,
        ):
            event_id = make_event_id(
                extraction.boe_id,
                event_idx,
            )

            records.append(
                {
                    "event_id": event_id,
                    "identificador_boe": extraction.boe_id,
                    "fecha_publicacion": extraction.publication_date,
                    "event_index": event_idx,
                    "event_type": enum_value(event.event_type),
                    "event_summary": event.event_summary,
                    "evidence": event.evidence,
                }
            )

    if not records:
        return empty_df(PUBLICATION_EVENTS_COLUMNS)

    return pd.DataFrame(
        records,
        columns=PUBLICATION_EVENTS_COLUMNS,
    )

#### flatten_administrative_actions

In [51]:
def flatten_administrative_actions(
    ai_extractions: pd.DataFrame,
) -> pd.DataFrame:
    """
    Aplana los actos administrativos publicados.

    Una fila representa un trámite, acto o decisión administrativa dentro de un
    `PublicationEvent`.
    """
    records = []

    for extraction in iter_valid_extractions(ai_extractions):
        for event_idx, event in enumerate(
            extraction.publication_events,
            start=1,
        ):
            event_id = make_event_id(
                extraction.boe_id,
                event_idx,
            )

            for action_idx, action in enumerate(
                event.administrative_actions,
                start=1,
            ):
                action_id = (
                    f"{event_id}"
                    f"_action_{action_idx}"
                )

                records.append(
                    {
                        "action_id": action_id,
                        "event_id": event_id,
                        "identificador_boe": extraction.boe_id,
                        "fecha_publicacion": extraction.publication_date,
                        "action_index": action_idx,
                        "procedure_stage": enum_value(action.procedure_stage),
                        "procedure_decision": enum_value(action.procedure_decision),
                        "evidence": action.evidence,
                    }
                )

    if not records:
        return empty_df(ADMINISTRATIVE_ACTIONS_COLUMNS)

    return pd.DataFrame(
        records,
        columns=ADMINISTRATIVE_ACTIONS_COLUMNS,
    )

#### flatten_project_mentions

In [52]:
def flatten_project_mentions(
    ai_extractions: pd.DataFrame,
) -> pd.DataFrame:
    """
    Aplana las menciones de proyectos o instalaciones energéticas.

    Una fila representa una mención sustantiva de proyecto dentro de un evento
    publicado. Todavía no es un proyecto consolidado.
    """
    records = []

    for extraction in iter_valid_extractions(ai_extractions):
        for event_idx, event in enumerate(
            extraction.publication_events,
            start=1,
        ):
            event_id = make_event_id(
                extraction.boe_id,
                event_idx,
            )

            for project in event.project_mentions:
                project_mention_id = make_project_mention_id(
                    event_id,
                    project.local_project_id,
                )

                records.append(
                    {
                        "project_mention_id": project_mention_id,
                        "event_id": event_id,
                        "identificador_boe": extraction.boe_id,
                        "fecha_publicacion": extraction.publication_date,
                        "local_project_id": project.local_project_id,
                        "project_name": project.name,
                        "project_name_norm": normalize_text_or_none(project.name),
                        "role_in_event": enum_value(project.role_in_event),
                        "status_in_document": enum_value(project.status_in_document),
                        "case_file_reference": project.case_file_reference,
                        "evidence": project.evidence,
                    }
                )

    if not records:
        return empty_df(PROJECT_MENTIONS_COLUMNS)

    return pd.DataFrame(
        records,
        columns=PROJECT_MENTIONS_COLUMNS,
    )

#### flatten_project_technical_attributes

In [53]:
def flatten_project_technical_attributes(
    ai_extractions: pd.DataFrame,
) -> pd.DataFrame:
    """
    Aplana las características técnicas de cada mención de proyecto.

    Cada fila representa un bloque técnico asociado a un tipo de
    instalación concreto.
    """
    records = []

    for extraction in iter_valid_extractions(ai_extractions):
        for event_idx, event in enumerate(
            extraction.publication_events,
            start=1,
        ):
            event_id = make_event_id(
                extraction.boe_id,
                event_idx,
            )

            for project in event.project_mentions:
                project_mention_id = make_project_mention_id(
                    event_id,
                    project.local_project_id,
                )

                for attribute_idx, attr in enumerate(
                    project.technical_attributes,
                    start=1,
                ):
                    records.append(
                        {
                            "project_technical_attribute_id": (
                                f"{project_mention_id}"
                                f"_attribute_{attribute_idx}"
                            ),
                            "project_mention_id": project_mention_id,
                            "event_id": event_id,
                            "identificador_boe": extraction.boe_id,
                            "fecha_publicacion": (
                                extraction.publication_date
                            ),
                            "attribute_index": attribute_idx,
                            "installation_type": enum_value(
                                attr.installation_type
                            ),
                            "power_mw": attr.power_mw,
                            "peak_power_mwp": attr.peak_power_mwp,
                            "storage_capacity_mwh": (
                                attr.storage_capacity_mwh
                            ),
                            "description": attr.description,
                            "power_normalization_note": (
                                attr.power_normalization_note
                            ),
                            "evidence": attr.evidence,
                        }
                    )

    if not records:
        return empty_df(
            PROJECT_TECHNICAL_ATTRIBUTES_COLUMNS
        )

    return pd.DataFrame(
        records,
        columns=PROJECT_TECHNICAL_ATTRIBUTES_COLUMNS,
    )

#### flatten_project_participants

In [54]:
def flatten_project_participants(
    ai_extractions: pd.DataFrame,
) -> pd.DataFrame:
    """
    Aplana participantes asociados a cada mención de proyecto.

    Incluye promotores, titulares, órganos administrativos u otras entidades
    extraídas explícitamente del BOE.
    """
    records = []

    for extraction in iter_valid_extractions(ai_extractions):
        for event_idx, event in enumerate(
            extraction.publication_events,
            start=1,
        ):
            event_id = make_event_id(
                extraction.boe_id,
                event_idx,
            )

            for project in event.project_mentions:
                project_mention_id = make_project_mention_id(
                    event_id,
                    project.local_project_id,
                )

                for participant_idx, participant in enumerate(
                    project.participants,
                    start=1,
                ):
                    participant_name_norm = normalize_text_or_none(
                        participant.name
                    )

                    if participant_name_norm is None:
                        continue

                    records.append(
                        {
                            "project_participant_id": (
                                f"{project_mention_id}"
                                f"_participant_{participant_idx}"
                            ),
                            "project_mention_id": project_mention_id,
                            "event_id": event_id,
                            "identificador_boe": extraction.boe_id,
                            "fecha_publicacion": (
                                extraction.publication_date
                            ),
                            "participant_index": participant_idx,
                            "participant_name": participant.name,
                            "participant_name_norm": (
                                participant_name_norm
                            ),
                            "participant_role": enum_value(
                                participant.role
                            ),
                            "evidence": participant.evidence,
                        }
                    )

    if not records:
        return empty_df(PROJECT_PARTICIPANTS_COLUMNS)

    return pd.DataFrame(
        records,
        columns=PROJECT_PARTICIPANTS_COLUMNS,
    )

#### flatten_project_locations

In [55]:
def flatten_project_locations(
    ai_extractions: pd.DataFrame,
) -> pd.DataFrame:
    """
    Aplana menciones textuales de localización.

    La IA solo extrae candidatos textuales. La resolución INE debe hacerse
    después mediante un proceso determinista.
    """
    records = []

    for extraction in iter_valid_extractions(ai_extractions):
        for event_idx, event in enumerate(
            extraction.publication_events,
            start=1,
        ):
            event_id = make_event_id(
                extraction.boe_id,
                event_idx,
            )

            for project in event.project_mentions:
                project_mention_id = make_project_mention_id(
                    event_id,
                    project.local_project_id,
                )

                for location_idx, loc in enumerate(
                    project.locations,
                    start=1,
                ):
                    municipality_raw = loc.municipality_name
                    municipality_raw_norm = normalize_text_or_none(
                        municipality_raw
                    )

                    if (
                        municipality_raw_norm is None
                        or municipality_raw_norm
                        in INVALID_LOCATION_VALUES
                    ):
                        continue

                    records.append(
                        {
                            "project_location_id": (
                                f"{project_mention_id}"
                                f"_location_{location_idx}"
                            ),
                            "project_mention_id": project_mention_id,
                            "event_id": event_id,
                            "identificador_boe": extraction.boe_id,
                            "fecha_publicacion": (
                                extraction.publication_date
                            ),
                            "location_index": location_idx,
                            "municipality_raw": municipality_raw,
                            "municipality_raw_norm": (
                                municipality_raw_norm
                            ),
                            "province_hint_raw": loc.province_hint,
                            "province_hint_raw_norm": (
                                normalize_text_or_none(
                                    loc.province_hint
                                )
                            ),
                            "autonomous_community_hint_raw": (
                                loc.autonomous_community_hint
                            ),
                            "autonomous_community_hint_raw_norm": (
                                normalize_text_or_none(
                                    loc.autonomous_community_hint
                                )
                            ),
                            "location_evidence": loc.evidence,
                        }
                    )

    if not records:
        return empty_df(PROJECT_LOCATIONS_COLUMNS)

    return pd.DataFrame(
        records,
        columns=PROJECT_LOCATIONS_COLUMNS,
    )

#### flatten_project_aliases

In [56]:
def flatten_project_aliases(
    ai_extractions: pd.DataFrame,
) -> pd.DataFrame:
    """
    Aplana alias o denominaciones alternativas de cada mención de proyecto.
    """
    records = []

    for extraction in iter_valid_extractions(ai_extractions):
        for event_idx, event in enumerate(
            extraction.publication_events,
            start=1,
        ):
            event_id = make_event_id(
                extraction.boe_id,
                event_idx,
            )

            for project in event.project_mentions:
                project_mention_id = make_project_mention_id(
                    event_id,
                    project.local_project_id,
                )

                for alias_idx, alias in enumerate(
                    project.aliases,
                    start=1,
                ):
                    alias_norm = normalize_text_or_none(alias)

                    if alias_norm is None:
                        continue

                    records.append(
                        {
                            "project_alias_id": (
                                f"{project_mention_id}"
                                f"_alias_{alias_idx}"
                            ),
                            "project_mention_id": project_mention_id,
                            "event_id": event_id,
                            "identificador_boe": extraction.boe_id,
                            "fecha_publicacion": extraction.publication_date,
                            "alias_index": alias_idx,
                            "alias": alias,
                            "alias_norm": alias_norm,
                        }
                    )

    if not records:
        return empty_df(PROJECT_ALIASES_COLUMNS)

    return pd.DataFrame(
        records,
        columns=PROJECT_ALIASES_COLUMNS,
    )

#### flatten_associated_infrastructure

In [57]:
def flatten_associated_infrastructure(
    ai_extractions: pd.DataFrame,
) -> pd.DataFrame:
    """
    Aplana el resumen de infraestructura asociada de cada evento publicado.

    Esta tabla no descompone líneas, subestaciones o posiciones eléctricas. Solo
    conserva la síntesis auxiliar extraída en `AssociatedInfrastructureSummary`.
    """
    records = []

    for extraction in iter_valid_extractions(ai_extractions):
        for event_idx, event in enumerate(
            extraction.publication_events,
            start=1,
        ):
            infra = event.associated_infrastructure

            if infra is None:
                continue

            has_content = any(
                [
                    infra.has_evacuation_infrastructure is not None,
                    infra.has_electrical_substation is not None,
                    infra.has_grid_connection is not None,
                    infra.has_shared_infrastructure is not None,
                    safe_str(infra.description) != "",
                    safe_str(infra.evidence) != "",
                ]
            )

            if not has_content:
                continue

            event_id = make_event_id(
                extraction.boe_id,
                event_idx,
            )

            records.append(
                {
                    "associated_infrastructure_id": (
                        f"{event_id}_associated_infrastructure"
                    ),
                    "event_id": event_id,
                    "identificador_boe": extraction.boe_id,
                    "fecha_publicacion": extraction.publication_date,
                    "has_evacuation_infrastructure": (
                        infra.has_evacuation_infrastructure
                    ),
                    "has_electrical_substation": (
                        infra.has_electrical_substation
                    ),
                    "has_grid_connection": infra.has_grid_connection,
                    "has_shared_infrastructure": infra.has_shared_infrastructure,
                    "description": infra.description,
                    "evidence": infra.evidence,
                }
            )

    if not records:
        return empty_df(ASSOCIATED_INFRASTRUCTURE_COLUMNS)

    return pd.DataFrame(
        records,
        columns=ASSOCIATED_INFRASTRUCTURE_COLUMNS,
    )

### Generar tablas derivadas

In [58]:
ai_extractions = load_ai_extractions(
    BOE_AI_EXTRACTIONS_PATH
)

publication_events = flatten_publication_events(ai_extractions)
administrative_actions = flatten_administrative_actions(ai_extractions)
project_mentions = flatten_project_mentions(ai_extractions)
project_technical_attributes = flatten_project_technical_attributes(ai_extractions)
project_participants = flatten_project_participants(ai_extractions)
project_locations = flatten_project_locations(ai_extractions)
project_aliases = flatten_project_aliases(ai_extractions)
associated_infrastructure = flatten_associated_infrastructure(ai_extractions)

### Guardar tablas derivadas

In [59]:
save_parquet(publication_events, PUBLICATION_EVENTS_PATH)
save_parquet(administrative_actions, ADMINISTRATIVE_ACTIONS_PATH)
save_parquet(project_mentions, PROJECT_MENTIONS_PATH)
save_parquet(project_technical_attributes, PROJECT_TECHNICAL_ATTRIBUTES_PATH)
save_parquet(project_participants, PROJECT_PARTICIPANTS_PATH)
save_parquet(project_locations, PROJECT_LOCATIONS_PATH)
save_parquet(project_aliases, PROJECT_ALIASES_PATH)
save_parquet(associated_infrastructure, ASSOCIATED_INFRASTRUCTURE_PATH)

### Recargar tablas derivadas

In [60]:
publication_events = pd.read_parquet(PUBLICATION_EVENTS_PATH)
administrative_actions = pd.read_parquet(ADMINISTRATIVE_ACTIONS_PATH)
project_mentions = pd.read_parquet(PROJECT_MENTIONS_PATH)
project_technical_attributes = pd.read_parquet(PROJECT_TECHNICAL_ATTRIBUTES_PATH)
project_participants = pd.read_parquet(PROJECT_PARTICIPANTS_PATH)
project_locations = pd.read_parquet(PROJECT_LOCATIONS_PATH)
project_aliases = pd.read_parquet(PROJECT_ALIASES_PATH)
associated_infrastructure = pd.read_parquet(ASSOCIATED_INFRASTRUCTURE_PATH)

### Inspección rápida

In [61]:
display(publication_events)
display(administrative_actions)
display(project_mentions)
display(project_technical_attributes)
display(project_participants)
display(project_locations)
display(project_aliases)
display(associated_infrastructure)

,event_id,identificador_boe,fecha_publicacion,event_index,event_type,event_summary,evidence
0,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,1,proyecto_nuevo,Sometimiento a información pública del Estudio...,Anuncio del Área de Industria y Energía de la ...
1,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,1,hibridacion,Formulación de declaración de impacto ambienta...,"Resolución de 22 de diciembre de 2022, de la D..."
2,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,1,proyecto_nuevo,Formulación de la declaración de impacto ambie...,"Resolución de 23 de enero de 2023, de la Direc..."
3,BOE-A-2023-10306_event_1,BOE-A-2023-10306,2023-04-28,1,proyecto_nuevo,Otorgamiento de autorización administrativa pr...,"Resolución de 17 de abril de 2023, de la Direc..."
4,BOE-B-2023-19082_event_1,BOE-B-2023-19082,2023-06-22,1,modificacion,Sometimiento a información pública de la modif...,Anuncio del Área Funcional de Industria y Ener...
5,BOE-A-2024-9608_event_1,BOE-A-2024-9608,2024-05-13,1,proyecto_nuevo,Formulación de informe de determinación de afe...,"Resolución de 6 de mayo de 2024, de la Direcci..."
6,BOE-B-2024-26379_event_1,BOE-B-2024-26379,2024-07-13,1,hibridacion,Sometimiento a información pública de la solic...,Anuncio del Área de Industria y Energía de la ...
7,BOE-A-2024-16664_event_1,BOE-A-2024-16664,2024-08-10,1,modificacion,Se otorga autorización administrativa previa d...,"Resolución de 22 de julio de 2024, de la Direc..."
8,BOE-A-2025-18285_event_1,BOE-A-2025-18285,2025-09-15,1,hibridacion,Se otorgan las autorizaciones administrativas ...,"Resolución de 7 de agosto de 2025, de la Direc..."
9,BOE-A-2025-26110_event_1,BOE-A-2025-26110,2025-12-19,1,proyecto_nuevo,Otorgamiento de autorización administrativa pr...,"Resolución de 17 de noviembre de 2025, de la D..."


,action_id,event_id,identificador_boe,fecha_publicacion,action_index,procedure_stage,procedure_decision,evidence
0,BOE-B-2021-32560_event_1_action_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,1,informacion_publica,sometido_informacion_publica,"se somete al trámite de información pública, d..."
1,BOE-B-2021-32560_event_1_action_2,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,2,solicitud_tramitacion_ambiental,solicitado,solicitud de Autorización Administrativa Previ...
2,BOE-B-2021-32560_event_1_action_3,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,3,solicitud_tramitacion,solicitado,solicitud de Autorización Administrativa Previ...
3,BOE-A-2022-24404_event_1_action_1,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,1,declaracion_impacto_ambiental,formulado,se formula la declaración de impacto ambiental...
4,BOE-A-2023-2598_event_1_action_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,1,declaracion_impacto_ambiental,formulado,"Resolución de 23 de enero de 2023, de la Direc..."
5,BOE-A-2023-10306_event_1_action_1,BOE-A-2023-10306_event_1,BOE-A-2023-10306,2023-04-28,1,solicitud_tramitacion,solicitado,"Enel Green Power España, SL, (en adelante, el ..."
6,BOE-A-2023-10306_event_1_action_2,BOE-A-2023-10306_event_1,BOE-A-2023-10306,2023-04-28,2,informacion_publica,sometido_informacion_publica,"Asimismo, la petición fue sometida a informaci..."
7,BOE-A-2023-10306_event_1_action_3,BOE-A-2023-10306_event_1,BOE-A-2023-10306,2023-04-28,3,autorizacion_administrativa_previa,autorizado,"Resolución de 17 de abril de 2023, de la Direc..."
8,BOE-B-2023-19082_event_1_action_1,BOE-B-2023-19082_event_1,BOE-B-2023-19082,2023-06-22,1,modificacion,sometido_informacion_publica,se somete a Información Pública la modificació...
9,BOE-B-2023-19082_event_1_action_2,BOE-B-2023-19082_event_1,BOE-B-2023-19082,2023-06-22,2,autorizacion_administrativa_construccion,solicitado,se somete a Información Pública (...) la solic...


,project_mention_id,event_id,identificador_boe,fecha_publicacion,local_project_id,project_name,project_name_norm,role_in_event,status_in_document,case_file_reference,evidence
0,BOE-B-2021-32560_event_1_project_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,project_1,Parque eólico Badulaque,parque eolico badulaque,objeto_principal,en_tramitacion,PEol-416,"Parque eólico Badulaque de 90 MW, accesos y su..."
1,BOE-A-2022-24404_event_1_project_1,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,project_1,Planta Fotovoltaica Híbrida Majal Alto,planta fotovoltaica hibrida majal alto,objeto_principal,en_tramitacion,None,"Planta fotovoltaica híbrida Majal Alto de 43,5..."
2,BOE-A-2022-24404_event_1_project_2,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,project_2,Parque Eólico Majal Alto,parque eolico majal alto,referencia_existente,en_explotacion,None,"Parque Eólico Majal Alto, ya operativo"
3,BOE-A-2023-2598_event_1_project_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,project_1,Parque eólico Badulaque,parque eolico badulaque,objeto_principal,en_tramitacion,None,"Parque eólico Badulaque de 90 MW, y su infraes..."
4,BOE-A-2023-10306_event_1_project_1,BOE-A-2023-10306_event_1,BOE-A-2023-10306,2023-04-28,project_1,parque eólico Badulaque,parque eolico badulaque,objeto_principal,en_tramitacion,None,parque eólico Badulaque de 90 MW
5,BOE-B-2023-19082_event_1_project_1,BOE-B-2023-19082_event_1,BOE-B-2023-19082,2023-06-22,project_1,Parque Eólico Badulaque,parque eolico badulaque,objeto_principal,en_tramitacion,PEol-416,"instalación «Parque Eólico Badulaque», de 102,..."
6,BOE-A-2024-9608_event_1_project_1,BOE-A-2024-9608_event_1,BOE-A-2024-9608,2024-05-13,project_1,Instalación solar FV La Puebla 1,instalacion solar fv la puebla 1,objeto_principal,en_tramitacion,None,Instalación solar FV La Puebla 1 de 100 MW de ...
7,BOE-A-2024-9608_event_1_project_2,BOE-A-2024-9608_event_1,BOE-A-2024-9608,2024-05-13,project_2,PSFV La Puebla 2,psfv la puebla 2,referencia_asociada,desconocido,None,"PSFV La Puebla 2, instalación del mismo promot..."
8,BOE-B-2024-26379_event_1_project_1,BOE-B-2024-26379_event_1,BOE-B-2024-26379,2024-07-13,project_1,BESS Hibridación FV Andévalo,bess hibridacion fv andevalo,objeto_principal,proyectado,PFOT-ALM-045,"instalación de almacenamiento por baterías ""BE..."
9,BOE-B-2024-26379_event_1_project_2,BOE-B-2024-26379_event_1,BOE-B-2024-26379,2024-07-13,project_2,FV Andévalo,fv andevalo,referencia_existente,existente,None,"parque solar fotovoltaico existente, ""FV Andév..."


,project_technical_attribute_id,project_mention_id,event_id,identificador_boe,fecha_publicacion,attribute_index,installation_type,power_mw,peak_power_mwp,storage_capacity_mwh,description,power_normalization_note,evidence
0,BOE-B-2021-32560_event_1_project_1_attribute_1,BOE-B-2021-32560_event_1_project_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,1,eolica,90.00,NaN,None,Parque eólico Badulaque de 90 MW de potencia n...,None,Parque eólico Badulaque de 90 MW de potencia n...
1,BOE-A-2022-24404_event_1_project_1_attribute_1,BOE-A-2022-24404_event_1_project_1,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,1,fotovoltaica,43.50,50.447,None,"8 recintos, 93.420 módulos fijos en una estruc...","La potencia nominal es de 43,5 MWn y la potenc...",Planta fotovoltaica. Estará compuesta por 8 re...
2,BOE-A-2022-24404_event_1_project_2_attribute_1,BOE-A-2022-24404_event_1_project_2,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,1,eolica,NaN,NaN,None,"Parque eólico ya operativo, asociado a la hibr...",None,"Parque Eólico Majal Alto, ya operativo"
3,BOE-A-2023-2598_event_1_project_1_attribute_1,BOE-A-2023-2598_event_1_project_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,1,eolica,90.00,NaN,None,Parque eólico integrado por 20 aerogeneradores...,None,Parque eólico Badulaque de 90 MW... El PE Badu...
4,BOE-A-2023-10306_event_1_project_1_attribute_1,BOE-A-2023-10306_event_1_project_1,BOE-A-2023-10306_event_1,BOE-A-2023-10306,2023-04-28,1,eolica,90.00,NaN,None,parque eólico Badulaque de 90 MW,None,parque eólico Badulaque de 90 MW
5,BOE-B-2023-19082_event_1_project_1_attribute_1,BOE-B-2023-19082_event_1_project_1,BOE-B-2023-19082_event_1,BOE-B-2023-19082,2023-06-22,1,eolica,102.40,NaN,None,Parque Eólico Badulaque de potencia instalada,None,"102,4 MW de potencia instalada"
6,BOE-A-2024-9608_event_1_project_1_attribute_1,BOE-A-2024-9608_event_1_project_1,BOE-A-2024-9608_event_1,BOE-A-2024-9608,2024-05-13,1,fotovoltaica,100.00,NaN,None,100 MW de potencia instalada,None,Instalación solar FV La Puebla 1 de 100 MW de ...
7,BOE-A-2024-9608_event_1_project_2_attribute_1,BOE-A-2024-9608_event_1_project_2,BOE-A-2024-9608_event_1,BOE-A-2024-9608,2024-05-13,1,fotovoltaica,NaN,NaN,None,instalación del mismo promotor objeto de otra ...,None,instalación del mismo promotor objeto de otra ...
8,BOE-B-2024-26379_event_1_project_1_attribute_1,BOE-B-2024-26379_event_1_project_1,BOE-B-2024-26379_event_1,BOE-B-2024-26379,2024-07-13,1,almacenamiento,26.36,NaN,None,Sistema de almacenamiento de energía eléctrica...,None,Sistema de almacenamiento de energía eléctrica...
9,BOE-B-2024-26379_event_1_project_2_attribute_1,BOE-B-2024-26379_event_1_project_2,BOE-B-2024-26379_event_1,BOE-B-2024-26379,2024-07-13,1,fotovoltaica,42.56,NaN,None,parque solar fotovoltaico existente,None,"parque solar fotovoltaico existente, ""FV Andév..."


,project_participant_id,project_mention_id,event_id,identificador_boe,fecha_publicacion,participant_index,participant_name,participant_name_norm,participant_role,evidence
0,BOE-B-2021-32560_event_1_project_1_participant_1,BOE-B-2021-32560_event_1_project_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,1,"ENEL GREEN POWER ESPAÑA, S.L.",enel green power espana s l,promotor,"Peticionario: ENEL GREEN POWER ESPAÑA, S.L., c..."
1,BOE-A-2022-24404_event_1_project_1_participant_1,BOE-A-2022-24404_event_1_project_1,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,1,Iberdrola Renovables Andalucía,iberdrola renovables andalucia,promotor,promovido por Iberdrola Renovables Andalucía
2,BOE-A-2022-24404_event_1_project_1_participant_2,BOE-A-2022-24404_event_1_project_1,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,2,Dirección General de Política Energética y Min...,direccion general de politica energetica y min...,organo_sustantivo,la Dirección General de Política Energética y ...
3,BOE-A-2023-2598_event_1_project_1_participant_1,BOE-A-2023-2598_event_1_project_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,1,Enel Green Power S.L.,enel green power s l,promotor,promovido por Enel Green Power S.L.
4,BOE-A-2023-2598_event_1_project_1_participant_2,BOE-A-2023-2598_event_1_project_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,2,Dirección General de Política Energética y Min...,direccion general de politica energetica y min...,organo_sustantivo,respecto de la que la Dirección General de Pol...
5,BOE-A-2023-2598_event_1_project_1_participant_3,BOE-A-2023-2598_event_1_project_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,3,Dirección General de Calidad y Evaluación Ambi...,direccion general de calidad y evaluacion ambi...,organo_ambiental,"Resolución de 23 de enero de 2023, de la Direc..."
6,BOE-A-2023-10306_event_1_project_1_participant_1,BOE-A-2023-10306_event_1_project_1,BOE-A-2023-10306_event_1,BOE-A-2023-10306,2023-04-28,1,"Enel Green Power España, SL",enel green power espana sl,promotor,"se otorga a Enel Green Power España, SL, autor..."
7,BOE-B-2023-19082_event_1_project_1_participant_1,BOE-B-2023-19082_event_1_project_1,BOE-B-2023-19082_event_1,BOE-B-2023-19082,2023-06-22,1,"ENEL GREEN POWER ESPAÑA, S.L. (unipersonal)",enel green power espana s l unipersonal,promotor,promovido por la mercantil «ENEL GREEN POWER E...
8,BOE-A-2024-9608_event_1_project_1_participant_1,BOE-A-2024-9608_event_1_project_1,BOE-A-2024-9608_event_1,BOE-A-2024-9608,2024-05-13,1,"Jinko Greenfield Spain 3, SL",jinko greenfield spain 3 sl,promotor,"promovido por Jinko Greenfield Spain 3, SL"
9,BOE-B-2024-26379_event_1_project_1_participant_1,BOE-B-2024-26379_event_1_project_1,BOE-B-2024-26379_event_1,BOE-B-2024-26379,2024-07-13,1,"IBERDROLA RENOVABLES ANDALUCÍA, S.A.",iberdrola renovables andalucia s a,promotor,"Peticionario: IBERDROLA RENOVABLES ANDALUCÍA, ..."


,project_location_id,project_mention_id,event_id,identificador_boe,fecha_publicacion,location_index,municipality_raw,municipality_raw_norm,province_hint_raw,province_hint_raw_norm,autonomous_community_hint_raw,autonomous_community_hint_raw_norm,location_evidence
0,BOE-B-2021-32560_event_1_project_1_location_1,BOE-B-2021-32560_event_1_project_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,1,As Pontes,as pontes,A Coruña,a coruna,None,None,"Municipios afectados: As Pontes, As Somozas, C..."
1,BOE-B-2021-32560_event_1_project_1_location_2,BOE-B-2021-32560_event_1_project_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,2,As Somozas,as somozas,A Coruña,a coruna,None,None,"Municipios afectados: As Pontes, As Somozas, C..."
2,BOE-B-2021-32560_event_1_project_1_location_3,BOE-B-2021-32560_event_1_project_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,3,Cedeira,cedeira,A Coruña,a coruna,None,None,"Municipios afectados: As Pontes, As Somozas, C..."
3,BOE-B-2021-32560_event_1_project_1_location_4,BOE-B-2021-32560_event_1_project_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,4,Cerdido,cerdido,A Coruña,a coruna,None,None,"Municipios afectados: As Pontes, As Somozas, C..."
4,BOE-B-2021-32560_event_1_project_1_location_5,BOE-B-2021-32560_event_1_project_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,5,Moeche,moeche,A Coruña,a coruna,None,None,"Municipios afectados: As Pontes, As Somozas, C..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,BOE-A-2026-13454_event_1_project_1_location_4,BOE-A-2026-13454_event_1_project_1,BOE-A-2026-13454_event_1,BOE-A-2026-13454,2026-06-20,4,Puebla de Guzmán,puebla de guzman,Huelva,huelva,None,None,"en los términos municipales de Alosno, Tharsis..."
64,BOE-A-2026-13454_event_1_project_2_location_1,BOE-A-2026-13454_event_1_project_2,BOE-A-2026-13454_event_1,BOE-A-2026-13454,2026-06-20,1,Alosno,alosno,Huelva,huelva,None,None,"en los términos municipales de Alosno, Tharsis..."
65,BOE-A-2026-13454_event_1_project_2_location_2,BOE-A-2026-13454_event_1_project_2,BOE-A-2026-13454_event_1,BOE-A-2026-13454,2026-06-20,2,Tharsis,tharsis,Huelva,huelva,None,None,"en los términos municipales de Alosno, Tharsis..."
66,BOE-A-2026-13454_event_1_project_2_location_3,BOE-A-2026-13454_event_1_project_2,BOE-A-2026-13454_event_1,BOE-A-2026-13454,2026-06-20,3,Cerro del Andévalo,cerro del andevalo,Huelva,huelva,None,None,"en los términos municipales de Alosno, Tharsis..."


,project_alias_id,project_mention_id,event_id,identificador_boe,fecha_publicacion,alias_index,alias,alias_norm
0,BOE-A-2022-24404_event_1_project_1_alias_1,BOE-A-2022-24404_event_1_project_1,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,1,Planta Fotovoltaica Majal Alto,planta fotovoltaica majal alto
1,BOE-A-2022-24404_event_1_project_1_alias_2,BOE-A-2022-24404_event_1_project_1,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,2,FV Majal Alto,fv majal alto
2,BOE-A-2023-2598_event_1_project_1_alias_1,BOE-A-2023-2598_event_1_project_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,1,PE Badulaque,pe badulaque
3,BOE-B-2023-19082_event_1_project_1_alias_1,BOE-B-2023-19082_event_1_project_1,BOE-B-2023-19082_event_1,BOE-B-2023-19082,2023-06-22,1,PROYECTO MODIFICADO mayo 2023,proyecto modificado mayo 2023
4,BOE-A-2024-9608_event_1_project_1_alias_1,BOE-A-2024-9608_event_1_project_1,BOE-A-2024-9608_event_1,BOE-A-2024-9608,2024-05-13,1,PSFV La Puebla 1,psfv la puebla 1
5,BOE-A-2025-26110_event_1_project_2_alias_1,BOE-A-2025-26110_event_1_project_2,BOE-A-2025-26110_event_1,BOE-A-2025-26110,2025-12-19,1,parque solar fotovoltaico «FV La Puebla 1»,parque solar fotovoltaico fv la puebla 1
6,BOE-A-2025-26110_event_1_project_2_alias_2,BOE-A-2025-26110_event_1_project_2,BOE-A-2025-26110_event_1,BOE-A-2025-26110,2025-12-19,2,Instalación solar FV La Puebla 1 de 100 MW de ...,instalacion solar fv la puebla 1 de 100 mw de ...
7,BOE-A-2025-26110_event_1_project_3_alias_1,BOE-A-2025-26110_event_1_project_3,BOE-A-2025-26110_event_1,BOE-A-2025-26110,2025-12-19,1,parque solar fotovoltaico «FV La Puebla 2»,parque solar fotovoltaico fv la puebla 2
8,BOE-A-2025-26110_event_1_project_3_alias_2,BOE-A-2025-26110_event_1_project_3,BOE-A-2025-26110_event_1,BOE-A-2025-26110,2025-12-19,2,Proyecto de instalación solar FV La Puebla 2 d...,proyecto de instalacion solar fv la puebla 2 d...


,associated_infrastructure_id,event_id,identificador_boe,fecha_publicacion,has_evacuation_infrastructure,has_electrical_substation,has_grid_connection,has_shared_infrastructure,description,evidence
0,BOE-B-2021-32560_event_1_associated_infrastruc...,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,True,True,True,True,Infraestructura de evacuación que incluye 5 lí...,su infraestructura de evacuación. La evacuació...
1,BOE-A-2022-24404_event_1_associated_infrastruc...,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,True,True,True,True,"Línea eléctrica soterrada de 20 kV y 5,61 km h...",Línea eléctrica de evacuación. La energía se e...
2,BOE-A-2023-2598_event_1_associated_infrastructure,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,True,True,True,True,Infraestructura de evacuación asociada al parq...,"y su infraestructura de evacuación, y de su in..."
3,BOE-A-2023-10306_event_1_associated_infrastruc...,BOE-A-2023-10306_event_1,BOE-A-2023-10306,2023-04-28,True,None,None,None,Infraestructuras de evacuación asociadas al pa...,y sus infraestructuras de evacuación
4,BOE-B-2023-19082_event_1_associated_infrastruc...,BOE-B-2023-19082_event_1,BOE-B-2023-19082,2023-06-22,True,False,False,False,accesos y su infraestructura de evacuación,sus accesos y su infraestructura de evacuación
5,BOE-A-2024-9608_event_1_associated_infrastructure,BOE-A-2024-9608_event_1,BOE-A-2024-9608,2024-05-13,True,True,True,True,Infraestructura de evacuación que incluye una ...,La energía producida evacuará a través de la n...
6,BOE-B-2024-26379_event_1_associated_infrastruc...,BOE-B-2024-26379_event_1,BOE-B-2024-26379,2024-07-13,True,True,None,None,Infraestructura de evacuación para el sistema ...,"su infraestructura de evacuación, en la provin..."
7,BOE-A-2024-16664_event_1_associated_infrastruc...,BOE-A-2024-16664_event_1,BOE-A-2024-16664,2024-08-10,True,False,True,False,"Infraestructuras de evacuación, incluyendo una...",y sus infraestructuras de evacuación; Desplaza...
8,BOE-A-2025-18285_event_1_associated_infrastruc...,BOE-A-2025-18285_event_1,BOE-A-2025-18285,2025-09-15,True,True,True,True,Infraestructura de evacuación del módulo BESS ...,"infraestructura de evacuación, consistente en ..."
9,BOE-A-2025-26110_event_1_associated_infrastruc...,BOE-A-2025-26110_event_1,BOE-A-2025-26110,2025-12-19,True,True,True,True,Líneas subterránea de 30 kV conectando los cen...,"y su infraestructura de evacuación, en Puebla ..."


In [62]:
# Chequeos básicos

print(f"{len(publication_events)=}")
print(f"{len(administrative_actions)=}")
print(f"{len(project_mentions)=}")
print(f"{len(project_technical_attributes)=}")
print(f"{len(project_participants)=}")
print(f"{len(project_locations)=}")
print(f"{len(project_aliases)=}")
print(f"{len(associated_infrastructure)=}")

len(publication_events)=13
len(administrative_actions)=26
len(project_mentions)=22
len(project_technical_attributes)=22
len(project_participants)=24
len(project_locations)=68
len(project_aliases)=9
len(associated_infrastructure)=13


In [63]:
# Ejemplo de revisión de localizaciones

project_locations.loc[
    project_locations["municipality_raw_norm"].str.contains(
        "pontes",
        na=False,
    )
]

,project_location_id,project_mention_id,event_id,identificador_boe,fecha_publicacion,location_index,municipality_raw,municipality_raw_norm,province_hint_raw,province_hint_raw_norm,autonomous_community_hint_raw,autonomous_community_hint_raw_norm,location_evidence
0,BOE-B-2021-32560_event_1_project_1_location_1,BOE-B-2021-32560_event_1_project_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,1,As Pontes,as pontes,A Coruña,a coruna,None,None,"Municipios afectados: As Pontes, As Somozas, C..."
13,BOE-A-2023-2598_event_1_project_1_location_6,BOE-A-2023-2598_event_1_project_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,6,As Pontes de García Rodríguez,as pontes de garcia rodriguez,A Coruña,a coruna,None,None,"Concellos de Valdoviño, Cerdido, Cerdeira, Moe..."
19,BOE-A-2023-10306_event_1_project_1_location_6,BOE-A-2023-10306_event_1_project_1,BOE-A-2023-10306_event_1,BOE-A-2023-10306,2023-04-28,6,As Pontés,as pontes,A Coruña,a coruna,None,None,"ubicados en Valdoviño, Cerdido, Cedeira, Moech..."
25,BOE-B-2023-19082_event_1_project_1_location_6,BOE-B-2023-19082_event_1_project_1,BOE-B-2023-19082_event_1,BOE-B-2023-19082,2023-06-22,6,As Pontes de García Rodríguez,as pontes de garcia rodriguez,A Coruña,a coruna,Galicia,galicia,"Valdoviño, Cedeira, Cerdido, Moeche, As Somoza..."
34,BOE-A-2024-16664_event_1_project_1_location_6,BOE-A-2024-16664_event_1_project_1,BOE-A-2024-16664_event_1,BOE-A-2024-16664,2024-08-10,6,As Pontes de García Rodríguez,as pontes de garcia rodriguez,A Coruña,a coruna,None,None,ubicados en los términos municipales de Valdov...


In [64]:

project_mentions = pd.read_parquet(PROJECT_MENTIONS_PATH)
display(project_mentions)

,project_mention_id,event_id,identificador_boe,fecha_publicacion,local_project_id,project_name,project_name_norm,role_in_event,status_in_document,case_file_reference,evidence
0,BOE-B-2021-32560_event_1_project_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,project_1,Parque eólico Badulaque,parque eolico badulaque,objeto_principal,en_tramitacion,PEol-416,"Parque eólico Badulaque de 90 MW, accesos y su..."
1,BOE-A-2022-24404_event_1_project_1,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,project_1,Planta Fotovoltaica Híbrida Majal Alto,planta fotovoltaica hibrida majal alto,objeto_principal,en_tramitacion,None,"Planta fotovoltaica híbrida Majal Alto de 43,5..."
2,BOE-A-2022-24404_event_1_project_2,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,project_2,Parque Eólico Majal Alto,parque eolico majal alto,referencia_existente,en_explotacion,None,"Parque Eólico Majal Alto, ya operativo"
3,BOE-A-2023-2598_event_1_project_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,project_1,Parque eólico Badulaque,parque eolico badulaque,objeto_principal,en_tramitacion,None,"Parque eólico Badulaque de 90 MW, y su infraes..."
4,BOE-A-2023-10306_event_1_project_1,BOE-A-2023-10306_event_1,BOE-A-2023-10306,2023-04-28,project_1,parque eólico Badulaque,parque eolico badulaque,objeto_principal,en_tramitacion,None,parque eólico Badulaque de 90 MW
5,BOE-B-2023-19082_event_1_project_1,BOE-B-2023-19082_event_1,BOE-B-2023-19082,2023-06-22,project_1,Parque Eólico Badulaque,parque eolico badulaque,objeto_principal,en_tramitacion,PEol-416,"instalación «Parque Eólico Badulaque», de 102,..."
6,BOE-A-2024-9608_event_1_project_1,BOE-A-2024-9608_event_1,BOE-A-2024-9608,2024-05-13,project_1,Instalación solar FV La Puebla 1,instalacion solar fv la puebla 1,objeto_principal,en_tramitacion,None,Instalación solar FV La Puebla 1 de 100 MW de ...
7,BOE-A-2024-9608_event_1_project_2,BOE-A-2024-9608_event_1,BOE-A-2024-9608,2024-05-13,project_2,PSFV La Puebla 2,psfv la puebla 2,referencia_asociada,desconocido,None,"PSFV La Puebla 2, instalación del mismo promot..."
8,BOE-B-2024-26379_event_1_project_1,BOE-B-2024-26379_event_1,BOE-B-2024-26379,2024-07-13,project_1,BESS Hibridación FV Andévalo,bess hibridacion fv andevalo,objeto_principal,proyectado,PFOT-ALM-045,"instalación de almacenamiento por baterías ""BE..."
9,BOE-B-2024-26379_event_1_project_2,BOE-B-2024-26379_event_1,BOE-B-2024-26379,2024-07-13,project_2,FV Andévalo,fv andevalo,referencia_existente,existente,None,"parque solar fotovoltaico existente, ""FV Andév..."
